# PDF → Markdown converter (v3)

Run the cells in order, top to bottom:

1. **Install** dependencies (once per runtime). Tesseract is a system package: it is installed with `apt-get` where that is available, and otherwise reported with instructions instead of failing silently.
2. **Definitions**: converter, QC and manifest logic. Defines functions only; nothing is converted here.
3. **Configuration**: input and output folders, OCR languages.
4. **Conversion**: one run over every PDF in the input folder.
5. **Audit**: random sample of converted pages, re-verified from the written files with the same verification model as the conversion (token loss, additions, reading order, block order, table placement, text layer). Reports are named by output identity, so two PDFs with the same name do not collide.

**Two separate claims.** Every output states them apart, in its front matter, in its Conversion Notes and in the manifest:

- `extraction`: was the text extracted and serialized? COMPLETE, LOW_CONFIDENCE (OCR below the confidence minimum) or INCOMPLETE (gaps).
- `verification`: was the extracted content checked against the source by an independent method? VERIFIED (every non-blank page and table checked, and passed), UNVERIFIED (nothing failed, but part of the content had no independent check: OCR pages, rotated pages, an unchecked text layer over a scan, a table whose placement could not be checked), DISPUTED (a check ran and disagreed) or FAILED (a check found content loss).

**Status and outputs.** One Markdown file per PDF, named by status: `name.md` (PASS = COMPLETE and VERIFIED), `name.UNVERIFIED.md` (COMPLETE, but not fully verified), `name.PARTIAL.md` (extraction not COMPLETE, or verification DISPUTED), `name.FAILED.md` (verification FAILED). With `overwrite=True` a new run replaces the previous output of the same PDF, whatever its status. "Same PDF" means the same source path, as recorded in the output's front matter: an existing file that belongs to another PDF, or that the converter did not write, is never replaced; the new output gets the next free name (`name_1.md`) and the manifest records it, with or without `overwrite`.

**What the QC verifies.** The Markdown body is built from PyMuPDF. Each converted page is then compared with the text that a second, independent extractor (pdfminer.six) reads from the same page:

- token coverage below `qc_threshold` (default 0.88) fails the document;
- the comparison runs in both directions: output text that the reference does not contain, beyond the same tolerance, makes the verification DISPUTED (status PARTIAL). It is reported as a disagreement between extractors, not as added content. Against the page text the output was itself built from, the tolerance is 1% in both directions: more than 1% of that text missing from the output fails the document, whatever `qc_threshold` says;
- **reading order, in two parts**:
  - *n-gram agreement* below `order_threshold` (default 0.70) marks it PARTIAL. It adapts to page length: 4-grams from 20 tokens, 3-grams from 8, 2-grams from 4, plus a direct sequence comparison on every page under 20 tokens. This measures how much text survived and in what local order; it is **insensitive to whole blocks moving** (a fully reversed page scores about 92%);
  - *block order* below `block_order_threshold` (default 0.80), or any two blocks whose reference spans interleave, marks it PARTIAL. Each paragraph of the output is located in the reference by a 6-gram that occurs there exactly once; the check then asks whether those positions increase. This is what catches paragraphs or columns displaced as blocks, which the n-gram comparison cannot see. Pages too repetitive to anchor at least `block_order_min_blocks` (default 2) paragraphs unambiguously are skipped rather than guessed, and the manifest shows no block-order figure for them. A page whose whole text is a single block is judged by n-grams only;
- pages without a text layer (OCR pages) have no independent reference: the document is UNVERIFIED at best, and the pages must reach a mean OCR confidence of `ocr_min_conf` (default 60), otherwise it is PARTIAL;
- where a text layer sits on top of a page scan (a scanned book that already carries OCR text), both extractors read that same layer, so it is compared with OCR of the page image instead: agreement below `layer_threshold` (default 0.70) marks the document PARTIAL. This needs `ocr=True`; set `verify_text_layer=False` to skip it;
- tables: detected with PyMuPDF's own table finder (`page.find_tables()`), which also finds tables drawn without full ruling (booktabs style, aligned columns). Only the grid is taken from it; the cell text is built from the words PyMuPDF locates inside each cell, and a word that falls outside every cell goes to the nearest one, so no word is dropped. Each rendered cell must then hold exactly the words of its grid cell; below `table_threshold` (default 0.95) of matching cells the document is PARTIAL. This check confirms that the table was written as detected. It does **not** confirm that the detected grid is the right one: there is no second table detector any more. On a page with tables, reading order is still checked on the prose around them. If the table finder fails on a page, the page is converted as prose, the failure is listed, and the document is UNVERIFIED. A page whose text still forms aligned columns outside every detected table is counted in `borderless_tables_suspected` and listed as a fidelity risk;
- a page that raises an error during conversion becomes a gap (`[Context gap: page N could not be converted]`, status PARTIAL); the other pages are converted normally;
- blank pages are noted and are not counted as gaps.

**Not verified, and stated as such:** rendered-page fidelity, excluded figures, whether a detected table grid is the true one, and (with the cell 4 setting below) text layers that sit on top of a page scan.

**Quality tiers.** `quality` selects `fast` (no table detection, no page markers, no heading detection), `balanced` (the default; tables and markers, no OCR) or `high` (also OCR). Cell 4 passes its options explicitly, so the tier only sets what is left as `None`.

## What changed in v3

| Change | Effect (measured on the same test sets as v2) |
|---|---|
| Cell 4 sets `verify_text_layer=False` | Books that already carry a text layer over a scan are no longer re-read with OCR (5 to 9 s per page in v2). They are converted from their own layer and reported UNVERIFIED instead of PASS. Set it back to `True` to restore the check |
| Tables come from PyMuPDF `find_tables()`; pdfplumber is removed | Tables without full ruling are now written as tables: booktabs table in a two-column paper, borderless tables and label/value lists, all with 100% cell agreement (v2: flattened to prose, reported PASS). Cost: the finder runs on every page, about 0.1 to 0.4 s per page (108-page mixed corpus without OCR: 20 s in v2, 63 s in v3). A table of contents with aligned page numbers is also written as a table. Needs PyMuPDF 2.1 or later (cell 1 installs it): from that version the finder uses PyMuPDF's layout analysis, and older versions find ruled tables only |
| Internal loss tolerance is 1% | A page that drops one paragraph in ten (1,760 → 1,585 tokens) is FAILED; v2 reported it PASS / VERIFIED. One token lost on a short page is FAILED |
| Errors are contained per page; no second PDF parser has to open the file | A file with a damaged cross-reference table that PyMuPDF can repair is converted (v2: no output). An error on one page of five leaves a gap on that page and converts the other four (v2: no output) |

**Not changed in v3 (open findings of the v2 audit):** lines of code listings and lists are still joined into paragraphs; the block-order check still disputes some correct two-column pages; fidelity risks still do not change the status; the audit cell keeps its reference cache between runs. On pages of glyph specimens the two extractors split text into tokens differently and the page is reported as a loss although every word is present.

In [ ]:
# 1. Install dependencies (run once per runtime).
import os
import shutil
import subprocess
import sys

subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q", "--upgrade", "PyMuPDF>=2.1", "pdfminer.six", "pyyaml", "pytesseract", "pillow"],
    check=True,
)

# OCR engine plus the language packs named in OCR_LANGUAGES (cell 3). Tesseract is
# a system package, not a pip package: apt-get on Debian/Colab, brew on macOS, an
# installer on Windows. A missing engine is reported here and again in every
# Conversion Notes, so scanned pages never fail silently.
packs = ["tesseract-ocr", "tesseract-ocr-eng", "tesseract-ocr-rus", "tesseract-ocr-spa"]
if shutil.which("apt-get") and hasattr(os, "geteuid") and os.geteuid() == 0:
    subprocess.run(["apt-get", "install", "-y", "-qq", *packs], check=False)
elif shutil.which("tesseract"):
    version = subprocess.run(["tesseract", "--version"], capture_output=True, text=True).stdout.splitlines()
    print("Tesseract present:", version[0] if version else "version unknown")
else:
    print(
        "Tesseract is not installed: OCR is unavailable and scanned pages will be\n"
        "reported as gaps, not converted. Install it with\n"
        "  Debian/Colab: apt-get install -y " + " ".join(packs) + "\n"
        "  macOS:        brew install tesseract tesseract-lang\n"
        "  Windows:      the UB Mannheim build, then put tesseract.exe on PATH"
    )

In [ ]:
"""Single-controller, tool-backed PDF -> Markdown converter.

v3: tables are detected with PyMuPDF's own table finder (ruled, partly ruled and
unruled tables; pdfplumber is no longer used), the output is held to a 1% loss
tolerance against the page text it was built from, and an error on one page is
contained to that page instead of losing the document.

v2: the hyphenation rule is linear in paragraph length, the audit parses its
reference once per document, and reading order is checked positionally (block
order) as well as by n-gram agreement."""
from __future__ import annotations
import warnings
import argparse, hashlib, importlib.metadata, math, re, shutil, subprocess, sys, tempfile, os, unicodedata, uuid
from collections import Counter
from dataclasses import dataclass
from datetime import date
from pathlib import Path

import pymupdf

# Ignore only the known malformed FontBBox warning from pdfminer.
# Keep all other pdfminer diagnostics visible.
import logging

class _FontBBoxWarningFilter(logging.Filter):
    def filter(self, record):
        return not (
            record.name == "pdfminer.pdffont"
            and "Could not get FontBBox" in record.getMessage()
        )

_pdfminer_font_logger = logging.getLogger("pdfminer.pdffont")
# Re-running this notebook cell must not accumulate duplicate filters.
for _filter in list(_pdfminer_font_logger.filters):
    if type(_filter).__name__ == "_FontBBoxWarningFilter":
        _pdfminer_font_logger.removeFilter(_filter)
_pdfminer_font_logger.addFilter(_FontBBoxWarningFilter())

import yaml

@dataclass
class Options:
    quality: str = 'balanced'
    ocr: bool | None = None
    images: bool | None = None
    markers: bool | None = None
    table_style: str = 'pipe'
    max_pages: int | None = None
    overwrite: bool = False
    qc_threshold: float = 0.88
    ocr_lang: str = 'rus+eng'
    order_threshold: float = 0.70
    ocr_min_conf: float = 60.0
    verify_text_layer: bool = True
    table_threshold: float = 0.95
    layer_threshold: float = 0.70
    # v2: flag table-like blocks that no detector can see (no ruling lines).
    detect_borderless: bool = True
    # v2: positional reading-order check (see block_order()).
    block_order_threshold: float = 0.80
    block_order_min_blocks: int = 2

    def resolve(self):
        self.quality = {'1':'fast','2':'balanced','3':'high','draft':'fast','standard':'balanced','fidelity':'high'}.get(self.quality,self.quality)
        if self.quality not in ('fast','balanced','high'): raise ValueError('invalid quality')
        if self.table_style not in ('pipe','html'): raise ValueError('invalid table style')
        if self.max_pages is not None and self.max_pages < 1: raise ValueError('max pages must be positive')
        if not 0 <= self.order_threshold <= 1: raise ValueError('order threshold must be within [0, 1]')
        if not 0 <= self.block_order_threshold <= 1: raise ValueError('block order threshold must be within [0, 1]')
        if self.block_order_min_blocks < 2: raise ValueError('block order minimum must be at least 2')
        if self.ocr is None: self.ocr = self.quality == 'high'
        if self.images is None: self.images = self.quality == 'high'
        if self.markers is None: self.markers = self.quality != 'fast'
        return self

# ---------------------------------------------------------------------------
# Text normalisation: ONE tokenizer and ONE line-end hyphenation rule are used
# by the converter, the internal QC, the independent QC and the audit cell.
# ---------------------------------------------------------------------------
LETTER = r'[^\W\d_]'          # any Unicode letter (covers ё, ñ, á, ...)
HYPHEN = '-\u2010\u2011'
_BREAK_LEFT = re.compile(rf'((?:{LETTER}+[{HYPHEN}])*{LETTER}{{2,}})[{HYPHEN}]$')
_BREAK_RIGHT = re.compile(rf'^({LETTER}{{2,}}(?:[{HYPHEN}]{LETTER}+)*)')
# A hyphenated word continues on the NEXT LINE: one line break, never a blank
# line, so a paragraph ending in a hyphen is not glued to the next paragraph.
_BREAK_IN_TEXT = re.compile(rf'((?:{LETTER}+[{HYPHEN}])*{LETTER}{{2,}})[{HYPHEN}][ \t]*\n[ \t]*({LETTER}{{2,}}(?:[{HYPHEN}]{LETTER}+)*)')
# v2: _BREAK_LEFT is anchored with [$], so a match can only end at the end of the
# string it is searched in. Searching the whole accumulated paragraph made
# join_line_break() quadratic (measured: 100 lines = 196 ms, 1600 lines = 18 s).
# Only the tail can take part in a match; 160 characters covers any real token.
_HYPHEN_TAIL = 160
_COMPOUND = re.compile(rf'{LETTER}+(?:[{HYPHEN}]{LETTER}+)+')
_WORD = re.compile(rf'{LETTER}+')
_SENTENCE_END = ('.', '!', '?', ':', ';', '\u2026')
_LIST_ITEM = re.compile(r'^(?:[\u2022\u2023\u25e6\u2043\u2219\u25aa\u25a0]\s*|\[\d{1,3}\]\s+|\(?\d{1,3}\)\s+)')

def _fold(text):
    return unicodedata.normalize('NFKC', str(text)).replace('\u00ad', '').replace('\u2010', '-').replace('\u2011', '-')

def comparable_tokens(text):
    """The single tokenizer. Hyphens between letters (with or without a line
    break) are removed, so 'configu-<newline>ration', 'noise-only' and
    'noiseonly' compare equal and hyphen-repair decisions never count as loss."""
    text = re.sub(rf'(?<={LETTER})-(?:[ \t]*\n[ \t]*)?(?={LETTER})', '', _fold(text))
    return re.findall(r"\w+(?:[.,]\d+)*", text.casefold(), re.UNICODE)

def words(text): return comparable_tokens(text)
def grams(w,n=4): return Counter(tuple(w[i:i+n]) for i in range(max(0,len(w)-n+1)))
def ratio(a,b):
    if not a: return 1.0
    return sum((a & b).values())/sum(a.values())

def hyphen_vocabulary(texts):
    """Document vocabulary used to decide whether a line-end hyphen is real.

    'compounds' holds hyphenated words seen UNBROKEN in the document
    ('noise-only'); 'words' holds every plain word."""
    compounds=set(); plain=set()
    for t in texts:
        t=_fold(t).casefold()
        compounds.update(_COMPOUND.findall(t)); plain.update(_WORD.findall(t))
    return {'compounds': compounds, 'words': plain}

def _keep_hyphen(left_word, right_word, vocab):
    """True when the document itself shows the hyphen belongs to the word."""
    if not vocab: return False
    lw=_fold(left_word).casefold(); rw=_fold(right_word).casefold()
    joined=lw.split('-')[-1]+rw.split('-')[0]
    return f'{lw}-{rw}' in vocab['compounds'] and joined not in vocab['words']

def join_line_break(left, right, vocab=None):
    """Join two consecutive lines of one paragraph (the single hyphenation rule).

    'configu-' + 'ration'  -> 'configuration'   (line-end hyphenation removed)
    'noise-'   + 'only'    -> 'noise-only'      if the document uses 'noise-only'
    'Wiener-'  + 'Hopf'    -> 'Wiener-Hopf'     (capitalised continuation: compound)
    '3-'       + '5 dB'    -> '3-5 dB'          (numeric range)
    anything else          -> joined with one space."""
    if re.search(rf'\d[{HYPHEN}]$', left) and right[:1].isdigit(): return left+right   # numeric range
    l=_BREAK_LEFT.search(left[-_HYPHEN_TAIL:]); r=_BREAK_RIGHT.match(right)
    if l and r:
        if not r.group(1)[0].islower() or _keep_hyphen(l.group(1), r.group(1), vocab):
            return left+right
        return left[:-1]+right
    return left+' '+right

def repair_hyphens(text, vocab=None):
    """Apply the same rule to raw text that still contains line breaks."""
    def fix(m):
        if not m.group(2)[0].islower() or _keep_hyphen(m.group(1), m.group(2), vocab):
            return m.group(1)+'-'+m.group(2)
        return m.group(1)+m.group(2)
    return _BREAK_IN_TEXT.sub(fix, text)

def norm(text, vocab=None):
    return re.sub(r'\s+', ' ', repair_hyphens(str(text), vocab)).strip()

_TAG_START = re.compile(r'<(?=[A-Za-z/!?])')

def md_escape(paragraph):
    """Escape the few constructs that would make a Markdown reader treat prose
    as markup: a '<' that could open an HTML tag or comment (hidden text), and
    a paragraph that starts like a heading, a quote or a horizontal rule.
    A plain inequality such as 'x < 3' is not touched."""
    p=_TAG_START.sub(r'\\<', paragraph)
    if re.match(r'#{1,6}(?:\s|$)', p) or p.startswith('>') or re.fullmatch(r'\s*([-*_=])(?:\s*\1){2,}\s*', p):
        p='\\'+p
    return p

def table_md(rows, style):
    rows=[[norm(str(v)) if v is not None else '' for v in r] for r in rows]
    if not rows: return ''
    w=max(map(len,rows))
    rows=[r+['']*(w-len(r)) for r in rows]
    if style=='html':
        from html import escape
        return '<table>\n'+ '\n'.join('<tr>' + ''.join('<td>'+escape(v)+'</td>' for v in r)+'</tr>' for r in rows) + '\n</table>'
    def esc(v): return _TAG_START.sub(r'\\<', v.replace('|','\\|').replace('\n',' '))
    return '\n'.join(['| '+' | '.join(map(esc,rows[0]))+' |','| '+' | '.join(['---']*w)+' |'] + ['| '+' | '.join(map(esc,r))+' |' for r in rows[1:]])

# ---------------------------------------------------------------------------
# Layout: lines -> reading order -> paragraphs. Every native-text page (with or
# without tables, one or two columns) goes through this one path.
# ---------------------------------------------------------------------------
def span_join(spans):
    """Concatenate the spans of one line. A space is inserted only where there
    is a visible horizontal gap, so a word split across spans (font change,
    superscript, formula) is not broken apart."""
    out=''; prev=None
    for s in sorted(spans, key=lambda s: s['bbox'][0]):
        t=s.get('text','')
        if not t: continue
        if prev is not None and out and not out.endswith(' ') and not t.startswith(' '):
            if s['bbox'][0]-prev['bbox'][2] > 0.2*max(prev.get('size',0), s.get('size',0), 1): out+=' '
        out+=t; prev=s
    return out.strip()

def page_lines(blocks):
    """Text lines of a page as elements: bbox, text, dominant size, largest size, bold."""
    items=[]
    for b in blocks:
        if b.get('type') != 0: continue
        for line in b.get('lines', []):
            spans=[s for s in line.get('spans', []) if s.get('text','').strip()]
            label=span_join(line.get('spans', []))
            if not spans or not label: continue
            weight=Counter(); bold=0; total=0
            for s in spans:
                n=len(s['text'].strip()); total+=n
                weight[round(s.get('size',0),1)]+=n
                if s.get('flags',0) & 16 or 'bold' in s.get('font','').lower(): bold+=n
            items.append({'kind':'text','bbox':tuple(line.get('bbox', b['bbox'])),'text':label,
                          'size':weight.most_common(1)[0][0],'max_size':max(s.get('size',0) for s in spans),
                          'bold':bold > 0.6*total})
    return items

def column_side(bbox, page_width):
    """'L' / 'R' for an element inside one column, 'S' for one spanning both
    columns, 'C' for a narrow element sitting on the centre line (a page number,
    or an equation number at the edge of a column)."""
    in_left = bbox[2] <= page_width * .53
    in_right = bbox[0] >= page_width * .47
    if in_left and in_right: return 'C'
    if in_left: return 'L'
    if in_right: return 'R'
    return 'S'

def flow_side(bbox, page_width):
    """Column an element is read with: as column_side(), but a narrow centred
    element ('C') is attached to the column its centre falls in."""
    side=column_side(bbox, page_width)
    if side!='C': return side
    return 'L' if (bbox[0]+bbox[2])/2 < page_width/2 else 'R'

def is_two_column(items, page_width, hint=None):
    """Two-column test for one page.

    Strict rule: at least 5 lines in each column. When the document as a whole
    is two-column (hint from document_columns), a page with a SHORT column -
    typically the last page - is still accepted if its right-hand lines start
    where the document's right column starts."""
    n_left=sum(column_side(e['bbox'], page_width)=='L' for e in items)
    right=[e for e in items if column_side(e['bbox'], page_width)=='R']
    mostly_columns=n_left+len(right) >= len(items)*.70
    if n_left >= 5 and len(right) >= 5 and mostly_columns: return True
    if not hint or not right or n_left < 1 or not mostly_columns: return False
    aligned=sum(abs(e['bbox'][0]/page_width-hint['right']) <= 0.04 for e in right)
    return aligned >= 1 and aligned >= 0.5*len(right)

def document_columns(doc, n_pages):
    """Document-level layout hint: where the right column starts (as a share of
    the page width) if at least 40% of the text pages are clearly two-column;
    otherwise None."""
    starts=[]; considered=0
    for i in range(n_pages):
        page=doc[i]; lines=page_lines(page.get_text('dict')['blocks'])
        if len(lines) < 10: continue
        considered+=1; width=page.rect.width
        if is_two_column(lines, width):
            starts.append(min(e['bbox'][0] for e in lines if column_side(e['bbox'], width)=='R')/width)
    if not starts or len(starts) < 0.4*considered: return None
    starts.sort()
    return {'right': starts[len(starts)//2], 'pages': len(starts)}

def merge_baselines(items, page_width, split):
    """Fuse lines that share a baseline. The column layout is decided before
    merging and, on a two-column page, lines are fused only within one column."""
    merged=[]; last_by_side={}
    for e in sorted(items, key=lambda e: (e['bbox'][1], e['bbox'][0])):
        key=flow_side(e['bbox'], page_width) if split else 'ALL'
        idx=last_by_side.get(key)
        if idx is not None:
            p=merged[idx]; pb=p['bbox']; bb=e['bbox']
            overlap=min(bb[3],pb[3])-max(bb[1],pb[1])
            if overlap/max(1,pb[3]-pb[1]) > 0.6:
                first,second=(p,e) if bb[0] > pb[0] else (e,p)
                big=p if len(p['text']) >= len(e['text']) else e
                merged[idx]={'kind':'text','bbox':(min(pb[0],bb[0]),min(pb[1],bb[1]),max(pb[2],bb[2]),max(pb[3],bb[3])),
                             'text':first['text']+' '+second['text'],'size':big['size'],
                             'max_size':max(p['max_size'],e['max_size']),'bold':big['bold']}
                continue
        merged.append(dict(e)); last_by_side[key]=len(merged)-1
    return merged

def reading_order(elements, page_width, split):
    """Order text lines and tables. On a two-column page, elements spanning both
    columns (title, wide table, footer) cut the page into horizontal bands; each
    band is read left column first, then right column."""
    top=lambda e: (e['bbox'][1], e['bbox'][0])
    if not split: return sorted(elements, key=top)
    raw={id(e): column_side(e['bbox'], page_width) for e in elements}
    column=[e for e in elements if raw[id(e)] in 'LR']
    body_top=min((e['bbox'][1] for e in column), default=0)
    body_bottom=max((e['bbox'][3] for e in column), default=0)
    def side(e):
        # A narrow centred element outside the column body is a header/footer
        # (treated as spanning); inside the body it belongs to the column it sits in.
        if raw[id(e)]!='C': return raw[id(e)]
        if e['bbox'][3] <= body_top or e['bbox'][1] >= body_bottom: return 'S'
        return 'L' if (e['bbox'][0]+e['bbox'][2])/2 < page_width/2 else 'R'
    spanning=sorted((e for e in elements if side(e)=='S'), key=top)
    rest=[e for e in elements if side(e)!='S']
    def band(group):
        return (sorted((e for e in group if side(e)=='L'), key=top) +
                sorted((e for e in group if side(e)=='R'), key=top))
    ordered=[]
    for s in spanning:
        above=[e for e in rest if e['bbox'][1] < s['bbox'][1]]
        rest=[e for e in rest if e['bbox'][1] >= s['bbox'][1]]
        ordered+=band(above); ordered.append(s)
    return ordered+band(rest)

def paragraphs(lines, vocab, page_width, split):
    """Group ordered text lines into paragraphs from geometry and typography.

    A new paragraph starts when the font size or weight changes (headings),
    at a list item, after extra vertical space, at a first-line indent, or
    after a short last line ending a sentence. Punctuation alone never
    starts a paragraph."""
    if not lines: return []
    key=(lambda e: flow_side(e['bbox'], page_width)) if split else (lambda e: 'ALL')
    height=lambda e: max(1.0, e['bbox'][3]-e['bbox'][1])
    tol=lambda e: max(3.0, 0.5*e['size'])
    edges={}
    for e in lines:
        lo,hi=edges.get(key(e), (e['bbox'][0], e['bbox'][2]))
        edges[key(e)]=(min(lo,e['bbox'][0]), max(hi,e['bbox'][2]))
    jump=lambda a,b: key(a)!=key(b) or b['bbox'][1] < a['bbox'][1]-0.5*height(a)
    gaps=sorted(b['bbox'][1]-a['bbox'][3] for a,b in zip(lines,lines[1:])
                if not jump(a,b) and b['bbox'][1]-a['bbox'][3] < 2*height(a))
    typical=gaps[len(gaps)//2] if gaps else 0.0
    ends=lambda e: e['text'].rstrip().endswith(_SENTENCE_END)
    style=lambda a,b: abs(a['size']-b['size']) > 0.6 or a['bold'] != b['bold']
    def indented(i):
        a,b=lines[i-1],lines[i]
        if not jump(a,b) and _LIST_ITEM.match(a['text']): return False   # hanging indent of a list item
        base=edges[key(b)][0] if jump(a,b) else a['bbox'][0]
        if b['bbox'][0]-base <= tol(b): return False
        nxt=lines[i+1] if i+1 < len(lines) and not jump(b,lines[i+1]) else None
        return nxt is None or nxt['bbox'][0] <= b['bbox'][0]-tol(b)/2
    starts=[True]+[False]*(len(lines)-1); uses_indent=0
    for i in range(1,len(lines)):
        a,b=lines[i-1],lines[i]
        if jump(a,b): continue
        if style(a,b) or _LIST_ITEM.match(b['text']) or b['bbox'][1]-a['bbox'][3] > typical+0.45*height(a):
            starts[i]=True
        elif indented(i):
            starts[i]=True; uses_indent+=1
        elif ends(a) and a['bbox'][2] < edges[key(a)][1]-1.5*a['size']:
            starts[i]=True
    for i in range(1,len(lines)):
        a,b=lines[i-1],lines[i]
        if not jump(a,b): continue
        # Column or flow change: a paragraph may continue across it.
        # Entering or leaving a spanning/centred element always starts a paragraph.
        if style(a,b) or _LIST_ITEM.match(b['text']) or (key(a)!=key(b) and 'S' in (key(a), key(b))): starts[i]=True
        elif uses_indent >= 2: starts[i]=indented(i)
        else: starts[i]=ends(a)
    out=[]
    for i,e in enumerate(lines):
        if starts[i]: out.append(e['text'])
        else: out[-1]=join_line_break(out[-1], e['text'], vocab)
    return [md_escape(p) for p in out]

def ocr_paragraphs(text, vocab=None):
    """Paragraphs from OCR text: blank lines separate paragraphs; lines inside a
    paragraph are joined with the same hyphenation rule as native text."""
    out=[]
    for chunk in re.split(r'\n\s*\n', text):
        chunk=norm(chunk, vocab)
        if chunk: out.append(md_escape(chunk))
    return out

def side_by_side_lines(lines, page_width, split):
    """Number of text lines (three words or more) that have another such line
    to their right, on the same baseline, across a gutter-sized gap, and that
    the resolved layout will fuse into one line: a sign of columns the layout
    step did not separate. Short fragments (equation numbers) are ignored."""
    lines=[e for e in lines if len(e['text'].split()) >= 3]
    key=(lambda e: flow_side(e['bbox'], page_width)) if split else (lambda e: 'ALL')
    count=0
    for a in lines:
        ab=a['bbox']; h=max(1.0, ab[3]-ab[1])
        if any(b is not a and key(a)==key(b) and b['bbox'][0] >= ab[2]+1.5*a['size'] and
               min(ab[3],b['bbox'][3])-max(ab[1],b['bbox'][1]) > 0.5*h for b in lines):
            count+=1
    return count

def table_token_coverage(rows, rendered):
    source=Counter(t for row in rows for cell in row if cell is not None for t in comparable_tokens(cell))
    return ratio(source, Counter(comparable_tokens(markdown_plain_text(rendered))))

def independent_page_texts(src, n_pages):
    """Independent reference: per-page text from pdfminer.six.

    pdfminer.six parses the PDF and analyses the layout (text boxes, columns,
    reading order) with its own code, so it shares no extraction step with the
    PyMuPDF text that the Markdown body is built from.
    Returns (texts, lines, error). texts[i] is the page text (None where
    unavailable). lines[i] lists the page's text lines in pdfminer reading
    order as (x0, top, x1, bottom, text), measured from the top-left corner of
    the page's MediaBox, so that table regions can be told from prose."""
    texts=[]; lines=[]; error=None
    try:
        from pdfminer.high_level import extract_pages as pdfminer_pages
        from pdfminer.layout import LTTextContainer, LTTextLine
        for layout in pdfminer_pages(str(src), maxpages=n_pages):
            height=layout.bbox[3]-layout.bbox[1]; page_lines_=[]
            for box in layout:
                if not isinstance(box, LTTextContainer): continue
                parts=[ln for ln in box if isinstance(ln, LTTextLine)] or [box]
                for ln in parts:
                    page_lines_.append((ln.x0, height-ln.y1, ln.x1, height-ln.y0, ln.get_text()))
            lines.append(page_lines_); texts.append(''.join(t[4] for t in page_lines_))
    except Exception as exc:
        error=f'{type(exc).__name__}: {(str(exc).strip().splitlines() or [""])[0][:160]}'
    texts=texts[:n_pages]+[None]*(n_pages-len(texts))
    lines=lines[:n_pages]+[None]*(n_pages-len(lines))
    return texts, lines, error

def independent_page_text(src, page_number):
    """Independent reference text for one 1-based page (used by the audit cell); None if unavailable."""
    try:
        from pdfminer.high_level import extract_text as pdfminer_text
        return pdfminer_text(str(src), page_numbers=[page_number-1])
    except Exception:
        return None

def markdown_plain_text(markdown):
    """Undo exactly what the converter adds to the document text, and nothing
    else: its own comments, gap lines, image links, table markup, heading
    marks and backslash escapes. Text such as 'x < a and b > y' is left intact."""
    from html import unescape
    text=re.sub(r'<table>.*?</table>', lambda m: unescape(re.sub(r'</?(?:table|tr|td)>', ' ', m.group(0))), markdown, flags=re.S)
    text=re.sub(r'<!-- (?:Page \d+|Blank page \d+|OCR page \d+ [^>]*?|Text layer of page \d+ \(not read by OCR\)) -->', ' ', text)
    text=re.sub(r'^\[(?:Context|QC) gap:.*\]\s*$', ' ', text, flags=re.M)
    text=re.sub(r'^!\[Image from page \d+\]\(<[^>]*>\)\s*$', ' ', text, flags=re.M)
    text=re.sub(r'^\s*\|(?:\s*:?-{3,}:?\s*\|)+\s*$', ' ', text, flags=re.M)
    text=re.sub(r'^#{1,6}\s+', '', text, flags=re.M)
    text=re.sub(r'\\([<>#|=*_-])', r'\1', text)
    return text.replace('|', ' ')

def allowed_tokens(n, threshold):
    """Number of tokens that may differ on a page of n tokens without breaching
    the configured criterion: floor((1 - threshold) * n).

    This is exactly 'coverage >= threshold' for every page length. There is no
    minimum allowance: at the default threshold of 0.88 a page of 8 tokens or
    fewer may not differ by a single token."""
    return max(0, math.floor((1 - threshold) * n + 1e-12))

def token_loss(reference_tokens, output_tokens, threshold):
    """Loss of reference tokens in the output, with the same criterion for
    pages of any length.

    Returns (coverage, missing, allowed). A page fails when missing > allowed,
    with allowed = floor((1 - threshold) * n), i.e. when coverage < threshold.
    Very short pages are not exempted."""
    ref=Counter(reference_tokens); n=sum(ref.values())
    missing=sum((ref-Counter(output_tokens)).values())
    return ((n-missing)/n if n else 1.0), missing, allowed_tokens(n, threshold)

# Internal checks compare the output with the page text it was built from. Both
# come from the same extractor, so there is no extractor disagreement to allow
# for: at most this share of a page's tokens may be missing from the output
# (loss) or absent from the page text (additions). The configured qc_threshold
# applies only to the comparison with the independent extractor.
INTERNAL_TOLERANCE = 0.01
INTERNAL_ADDITION_TOLERANCE = INTERNAL_TOLERANCE

def token_additions(output_tokens, reference_tokens, threshold):
    """The reverse direction of token_loss(): output tokens that the reference
    does not contain (counting repeats, so duplicated text shows up too).

    Returns (share_found, extra, allowed, examples). The same criterion
    applies, measured on the output: extra > floor((1 - threshold) * n_output)
    is reported, with no minimum allowance. Additions are reported as a DISAGREEMENT, not as fabrication:
    two PDF extractors can legitimately differ, so text that only one of them
    sees is a dispute to be looked at, not proof of invented content."""
    out=Counter(output_tokens); n=sum(out.values())
    surplus=out-Counter(reference_tokens); extra=sum(surplus.values())
    return ((n-extra)/n if n else 1.0), extra, allowed_tokens(n, threshold), [t for t,_ in surplus.most_common(6)]

def loss_text(coverage, missing, n, threshold):
    """Wording of a token-coverage failure; the counts are always shown."""
    return f'{coverage:.1%} (< {threshold:.1%}): {missing} of {n} reference tokens missing, {allowed_tokens(n, threshold)} allowed'

def longest_common_subsequence(a, b):
    """Length of the longest common subsequence of two token lists."""
    previous=[0]*(len(b)+1)
    for x in a:
        current=[0]
        for j,y in enumerate(b,1):
            current.append(previous[j-1]+1 if x==y else max(previous[j], current[-1]))
        previous=current
    return previous[-1]

def order_agreement(reference_tokens, output_tokens, order_threshold=0.70, qc_threshold=0.88):
    """Length-aware reading-order agreement between a reference and the output.

    Returns None when there is nothing to order (fewer than two reference
    tokens), otherwise a dict:
      'score', 'minimum', 'method'   the n-gram comparison (or, for 2-3 tokens,
                                     the sequence comparison)
      'sequence', 'sequence_min'     the direct sequence comparison, used for
                                     every passage under 20 tokens
      'ok'                           False when a score is below its minimum

    The comparison adapts to the length of the reference, so short passages
    are checked instead of being skipped:

      20 tokens or more   share of reference 4-grams found in the output
      8 to 19 tokens      share of reference 3-grams, and sequence comparison
      4 to 7 tokens       share of reference 2-grams, and sequence comparison
      2 or 3 tokens       sequence comparison only

    Sequence comparison = longest common subsequence / reference tokens. On a
    short passage it catches what n-grams tolerate: two lines exchanged break
    only the n-grams at the join, but halve the common subsequence. It is not
    used on long pages, where extractors legitimately order whole blocks
    (headers, footnotes, captions) differently.

    Minimums: the configured order_threshold, lowered on short passages by
    what the token criterion already tolerates - allowed_tokens(n) differing
    tokens can break at most g n-grams each, and shorten the common
    subsequence by one each.

    What this cannot see: n-gram agreement is a set comparison, so moving whole
    blocks leaves every intra-block n-gram intact. Reversing all paragraphs of a
    page measurably scores 91.8% here. Block displacement is caught by
    block_order() instead, which compares positions rather than gram sets."""
    n=len(reference_tokens)
    if n < 2: return None
    tolerated=allowed_tokens(n, qc_threshold)
    result={'sequence': None, 'sequence_min': None}
    if n < 20:
        result['sequence']=longest_common_subsequence(reference_tokens, output_tokens)/n
        result['sequence_min']=max(0.0, min(order_threshold, 1-tolerated/n))
    if n < 4:
        result.update(score=result['sequence'], minimum=result['sequence_min'], method='sequence')
    else:
        g=4 if n >= 20 else 3 if n >= 8 else 2
        minimum=order_threshold if n >= 20 else max(0.0, min(order_threshold, 1-g*tolerated/(n-g+1)))
        result.update(score=ratio(grams(reference_tokens, g), grams(output_tokens, g)), minimum=minimum,
                      method=f'{g}-gram' if n >= 20 else f'{g}-gram + sequence')
    result['ok']=(result['score'] >= result['minimum']-1e-9 and
                  (result['sequence'] is None or result['sequence'] >= result['sequence_min']-1e-9))
    return result

def block_order(reference_tokens, output_markdown, threshold=0.80, min_blocks=2, window=6):
    """Positional reading-order check: are the output's blocks in the reference's order?

    order_agreement() compares n-gram sets, which cannot see whole blocks move:
    permuting paragraphs preserves every intra-block n-gram (91.8% agreement on a
    fully reversed page). This check locates each block of the output in the
    reference by a 6-gram that occurs there exactly once, then asks two questions:

      'score'     share of adjacent blocks whose reference position increases
                  (1.0 in order, 0.0 reversed)                  -> displaced blocks
      'overlaps'  block pairs whose reference spans interleave  -> interleaved columns

    Both are needed: interleaving two columns keeps 5 of 6 adjacent pairs in
    order but makes every span overlap. Blocks without an unambiguous anchor are
    skipped, so a repetitive page yields None instead of a guess.
    """
    blocks=[]
    for chunk in re.split(r'\n\s*\n', markdown_plain_text(output_markdown)):
        tokens=comparable_tokens(chunk)
        if len(tokens) >= window: blocks.append(tokens)
    if len(reference_tokens) < 2*window or len(blocks) < min_blocks: return None
    where={}
    for i in range(len(reference_tokens)-window+1):
        where.setdefault(tuple(reference_tokens[i:i+window]), []).append(i)
    spans=[]
    for tokens in blocks:
        hits=[where[key][0] for j in range(len(tokens)-window+1)
              if (key := tuple(tokens[j:j+window])) in where and len(where[key]) == 1]
        if hits: spans.append((min(hits), max(hits)))
    if len(spans) < min_blocks: return None
    starts=[a for a,_ in spans]
    up=sum(b >= a for a,b in zip(starts, starts[1:]))
    overlaps=0
    for i in range(len(spans)):
        for j in range(i+1, len(spans)):
            (a0,a1),(b0,b1)=spans[i],spans[j]
            if a0 < b1 and b0 < a1: overlaps+=1
    score=up/(len(starts)-1)
    return {'score': score, 'blocks': len(spans), 'overlaps': overlaps, 'threshold': threshold,
            'ok': score >= threshold-1e-9 and overlaps == 0}

def parse_rendered_table(rendered):
    """Read a rendered table (pipe or HTML style) back into a grid of cell texts,
    so that the structure check judges what was actually written."""
    if rendered.startswith('<table>'):
        from html import unescape
        return [[unescape(c) for c in re.findall(r'<td>(.*?)</td>', row, flags=re.S)]
                for row in re.findall(r'<tr>(.*?)</tr>', rendered, flags=re.S)]
    grid=[]
    for k,line in enumerate(rendered.split('\n')):
        if k==1: continue                       # the |---|---| separator
        grid.append([markdown_plain_text(c).strip() for c in re.split(r'(?<!\\)\|', line.strip())[1:-1]])
    return grid

def table_structure_check(page, table, rendered, to_fitz):
    """Verify row and column placement of a rendered table.

    The rendered cell at row r, column c must contain exactly the words that
    PyMuPDF locates inside the geometric cell (r, c) of the detected grid.
    This checks what was WRITTEN (the rendered table, read back) against where
    the words are on the page, so a swapped column, a shifted row or a cell
    filled from its neighbour shows up as a mismatch. Cell text and word
    positions both come from PyMuPDF: the check guards the rendering, it is
    not a second opinion on the table. The table's words are also covered by
    the page-level comparison with the independent extractor.

    Returns {'state': 'checked', 'cells', 'matching', 'agreement', 'wide_gap'}
    or {'state': 'unavailable'} when the two extractors' geometry does not line
    up (for example a rotated page). 'wide_gap' counts cells whose words are
    separated by a column-sized gap: evidence that the detected grid merged two
    columns, which cell-by-cell comparison cannot see."""
    grid=parse_rendered_table(rendered)
    box=to_fitz(table.bbox)
    centre=lambda w: ((w[0]+w[2])/2, (w[1]+w[3])/2)
    within=lambda w,b: b[0]-1 <= centre(w)[0] <= b[2]+1 and b[1]-1 <= centre(w)[1] <= b[3]+1
    inside=[w for w in page.get_text('words') if within(w, box)]
    def tokens_of(ws):
        ws=sorted(ws, key=lambda w: (w[5], w[6], w[7])); text=''; last=None
        for w in ws:
            text+=('' if last is None else ' ' if (w[5],w[6])==last else '\n')+w[4]; last=(w[5],w[6])
        return Counter(comparable_tokens(text))
    rendered_all=Counter(t for row in grid for c in row for t in comparable_tokens(c))
    if not rendered_all: return {'state':'unavailable'}
    if ratio(rendered_all, tokens_of(inside)) < 0.5: return {'state':'unavailable'}
    checked=matching=wide=0
    rows=table.rows
    for r in range(max(len(rows), len(grid))):
        cells=rows[r].cells if r < len(rows) else []
        texts=grid[r] if r < len(grid) else []
        for c in range(max(len(cells), len(texts))):
            cell=cells[c] if c < len(cells) else None
            ws=[w for w in inside if within(w, to_fitz(cell))] if cell else []
            located=tokens_of(ws); written=Counter(comparable_tokens(texts[c] if c < len(texts) else ''))
            if not located and not written: continue
            checked+=1; matching+=located==written
            ws=sorted(ws, key=lambda w: w[0])
            wide+=any(b[0]-a[2] > 2.5*max(1.0, a[3]-a[1]) and min(a[3],b[3])-max(a[1],b[1]) > 0.5*(a[3]-a[1]) for a,b in zip(ws, ws[1:]))
    return {'state':'checked','cells':checked,'matching':matching,'agreement':matching/checked if checked else 1.0,'wide_gap':wide}

def borderless_table_blocks(page, exclude=(), min_rows=3, min_cells=3, min_gap=12.0, x_tol=8.0):
    """Count table-like blocks that no table detector can see.

    A table finder can miss a table drawn without ruling lines: its cells are
    then flattened into prose and nothing about them is table-verified. This
    looks, outside the tables that were detected, for the geometry
    such a table leaves behind - at least min_rows consecutive text lines, each
    holding min_cells or more short cells separated by a gutter of at least
    min_gap points, with cell starts aligned within x_tol and at least one value
    (a digit) in every row.

    It is a hint used to raise a fidelity risk, never a reason to change the
    output: the words were extracted and are verified as prose.
    """
    def in_box(w, boxes):
        cx, cy = (w[0]+w[2])/2, (w[1]+w[3])/2
        return any(b[0] <= cx <= b[2] and b[1] <= cy <= b[3] for b in boxes)
    words=[w for w in page.get_text('words') if not in_box(w, exclude)]
    if len(words) < min_rows*min_cells: return 0
    # Visual rows, not PDF blocks: a generator may put every cell in its own block.
    rows=[]
    for w in sorted(words, key=lambda w: (w[1], w[0])):
        if rows and abs((rows[-1][0][1]+rows[-1][0][3])/2 - (w[1]+w[3])/2) <= 2.0: rows[-1].append(w)
        else: rows.append([w])
    def starts_of(ws):
        ws=sorted(ws, key=lambda w: w[0]); groups=[[ws[0]]]
        for w in ws[1:]:
            if w[0]-groups[-1][-1][2] > min_gap: groups.append([w])
            else: groups[-1].append(w)
        if len(groups) < min_cells or any(len(g) > 5 for g in groups): return None
        if not any(any(ch.isdigit() for ch in w[4]) for g in groups for w in g): return None
        return [g[0][0] for g in groups]
    blocks=0; run=[]
    def close(run):
        return 1 if len(run) >= min_rows else 0
    for ws in rows:
        starts=starts_of(ws)
        if starts is None:
            blocks+=close(run); run=[]; continue
        if run and (len(starts) != len(run[-1]) or any(abs(a-b) > x_tol for a,b in zip(starts, run[-1]))):
            blocks+=close(run); run=[]
        run.append(starts)
    return blocks+close(run)

def find_page_tables(page):
    """Tables of a page, from PyMuPDF's own table finder.

    Returns (tables, error). The finder works in PyMuPDF page coordinates and
    detects fully ruled tables, tables with horizontal rules only, and tables
    without any ruling (aligned columns of text). A 'table' with fewer than two
    rows or two columns is not a table and is ignored. Any failure, including a
    PyMuPDF release without the finder, yields no tables and the reason."""
    try:
        with warnings.catch_warnings():
            warnings.simplefilter('ignore')   # numeric warnings of the table finder, not conversion problems
            found=[t for t in page.find_tables().tables if t.row_count >= 2 and t.col_count >= 2]
        return found, None
    except Exception as exc:
        return [], f'{type(exc).__name__}: {(str(exc).strip().splitlines() or [""])[0][:120]}'

def table_cells(page, table):
    """Cell texts of a detected table, row by row, built from the page's words.

    The finder's own cell extraction can scramble characters inside a cell
    (measured: '98.1' returned as '981 .'), so only its GRID is used: every
    word of the page that lies in the table is placed in the cell that contains
    its centre, in reading order. A word of the table that falls in no cell (a
    merged or missing cell) goes to the nearest cell, so no text is dropped."""
    box=tuple(table.bbox)
    centre=lambda w: ((w[0]+w[2])/2, (w[1]+w[3])/2)
    inside=lambda w,b: b[0]-1 <= centre(w)[0] <= b[2]+1 and b[1]-1 <= centre(w)[1] <= b[3]+1
    words=[w for w in page.get_text('words') if inside(w, box)]
    grid=[[tuple(c) if c else None for c in row.cells] for row in table.rows]
    held={}
    cells=[(r,c,b) for r,row in enumerate(grid) for c,b in enumerate(row) if b]
    for w in words:
        home=next(((r,c) for r,c,b in cells if inside(w, b)), None)
        if home is None and cells:
            x,y=centre(w)
            gap=lambda b: max(b[0]-x, 0, x-b[2])+max(b[1]-y, 0, y-b[3])
            r,c,_=min(cells, key=lambda t: gap(t[2])); home=(r,c)
        if home is not None: held.setdefault(home, []).append(w)
    def text(ws):
        out=''; last=None
        for w in sorted(ws, key=lambda w: (w[5], w[6], w[7])):
            out+=('' if last is None else ' ' if (w[5],w[6])==last else '\n')+w[4]; last=(w[5],w[6])
        return out
    return [[text(held.get((r,c), [])) for c in range(len(row))] for r,row in enumerate(grid)]

def reference_shift(page):
    """(dx, dy) that brings a line of the independent reference - measured from
    the top-left corner of the page's MediaBox - into PyMuPDF page coordinates,
    which are measured from the CropBox corner."""
    try:
        position=page.cropbox_position
        return -(position.x-page.mediabox.x0), -position.y
    except Exception:
        return 0.0, 0.0

def independent_check(page, threshold=0.88, order_threshold=0.70, block_threshold=0.80, block_min_blocks=2):
    """Compare one converted page with the independent reference.

    Returns {'state', 'token', 'order', 'missing', 'allowed', 'n'}; state is
      'checked'      the token loss was measured (every page with reference
                     text, however short), together with the reading-order
                     comparison, whose method depends on the page length
                     (see order_agreement),
      'no_reference' the page has output text but the independent extractor
                     has none to compare with (scanned / OCR page, or
                     extractor failure),
      'empty'        neither side has any text.
    token: share of reference tokens present in the output (bag of tokens):
           measures LOSS.
    found: share of output tokens present in the reference: measures
           ADDITIONS (text only the converter's extractor sees, duplicated
           text, or text that is in neither source).
    order: reading-order agreement from order_agreement(); 'order_detail' holds
           the method, the minimums and the sequence score. On a
           page with tables it is computed on the prose around the tables:
           table text is removed from both sides (the tables themselves are
           covered by table_structure_check). It is None only when there is
           nothing to order (a single token), or when the reference prose
           could not be separated from the table text."""
    ref=comparable_tokens(page.get('independent') or '')
    out=comparable_tokens(markdown_plain_text(page['body']))
    if not ref or page.get('scanned'):
        return {'state': 'no_reference' if out else 'empty', 'token': None, 'order': None}
    token, missing, allowed = token_loss(ref, out, threshold)
    found, extra, extra_allowed, examples = token_additions(out, ref, threshold)
    order_ref, order_out = ref, out
    order_body=page['body']
    if page.get('tables'):
        prose=page.get('independent_prose')
        body=page['body']
        for _, rendered, _ in page['tables']: body=body.replace(rendered, ' ')
        order_body=body
        order_ref=comparable_tokens(prose) if prose is not None else None
        order_out=comparable_tokens(markdown_plain_text(body))
    agreement=order_agreement(order_ref, order_out, order_threshold, threshold) if order_ref is not None else None
    blocks=block_order(order_ref, order_body, block_threshold, block_min_blocks) if order_ref is not None else None
    return {'state': 'checked', 'token': token, 'order': agreement['score'] if agreement else None, 'order_detail': agreement,
            'missing': missing, 'allowed': allowed, 'n': len(ref),
            'found': found, 'extra': extra, 'extra_allowed': extra_allowed, 'examples': examples, 'n_out': len(out),
            'blocks': blocks, 'order_skipped': bool(page.get('tables')) and order_ref is None}

def ocr_setup(opt):
    """Resolve the OCR engine once per run.

    Returns {'lang': usable '+'-joined languages ('' if OCR cannot run),
             'missing': requested languages without an installed pack,
             'error': reason OCR cannot run, or None}.
    """
    if not opt.ocr:
        return {'lang': '', 'missing': [], 'error': '[OCR Fallback] = off'}
    if not shutil.which('tesseract'):
        return {'lang': '', 'missing': [], 'error': 'Tesseract is not installed'}
    wanted=[x for x in opt.ocr_lang.split('+') if x]
    try:
        import pytesseract
        installed=set(pytesseract.get_languages(config=''))
    except Exception as exc:
        return {'lang': '', 'missing': wanted, 'error': f'Tesseract language query failed ({type(exc).__name__})'}
    usable=[x for x in wanted if x in installed]
    missing=[x for x in wanted if x not in installed]
    if not usable:
        return {'lang': '', 'missing': missing, 'error': 'no installed Tesseract language pack among: '+'+'.join(wanted)}
    return {'lang': '+'.join(usable), 'missing': missing, 'error': None}

def run_ocr_on_page(page, lang):
    """OCR one page with a single Tesseract pass.

    Returns (text, mean_confidence, error); error is None on success. The text
    is rebuilt from the same word list the confidence is computed from:
    words -> lines (newline) -> paragraphs (blank line), using Tesseract's own
    block / paragraph / line numbering. A failure is reported to the caller
    instead of being returned as empty text."""
    import io
    try:
        import pytesseract
        from PIL import Image
        from pytesseract import Output
        pix=page.get_pixmap(matrix=pymupdf.Matrix(2,2),alpha=False)
        img=Image.open(io.BytesIO(pix.tobytes('png')))
        data=pytesseract.image_to_data(img,lang=lang,output_type=Output.DICT)
        confs=[]; paragraphs_=[]; last_par=None; last_line=None
        for k,word in enumerate(data['text']):
            if not word.strip(): continue
            if float(data['conf'][k]) >= 0: confs.append(float(data['conf'][k]))
            par=(data['block_num'][k], data['par_num'][k]); line=par+(data['line_num'][k],)
            if par != last_par: paragraphs_.append([[]])
            elif line != last_line: paragraphs_[-1].append([])
            paragraphs_[-1][-1].append(word.strip()); last_par=par; last_line=line
        text='\n\n'.join('\n'.join(' '.join(line) for line in par) for par in paragraphs_)
        ocr_conf=round(sum(confs)/len(confs),1) if confs else 0.0
        return text, ocr_conf, None
    except Exception as exc:
        detail=(str(exc).strip().splitlines() or [''])[0][:160]
        return '', None, f'{type(exc).__name__}: {detail}'

# A page is "image-backed" when raster images cover at least this share of it.
IMAGE_BACKED = 0.50
# An image-backed page's text layer is "sparse" when it holds fewer tokens than
# this share of what OCR reads on the page image.
SPARSE_LAYER = 0.50

def image_coverage(page):
    """Share of the page area covered by raster images (1.0 = a full-page scan)."""
    area=abs(page.rect)
    if not area: return 0.0
    total=0.0
    try:
        for info in page.get_image_info():
            r=pymupdf.Rect(info['bbox']) & page.rect
            if not r.is_empty: total+=abs(r)
    except Exception:
        return 0.0
    return min(1.0, total/area)

def text_agreement(a, b):
    """Bag-of-tokens agreement of two texts, penalising both missing and extra
    tokens: shared tokens / tokens of the longer text."""
    ta=Counter(comparable_tokens(a)); tb=Counter(comparable_tokens(b))
    return sum((ta & tb).values())/max(sum(ta.values()), sum(tb.values()), 1)

def extract_pages(doc, src, opt, asset_dir, stem):
    pages=[]; all_risks=[]; transform=[]; gap_entries=[]; image_count=0; tables_count=0; borderless_count=0; title=None
    ocr=ocr_setup(opt)
    if opt.ocr and ocr['error']: transform.append('OCR unavailable: '+ocr['error'])
    elif opt.ocr:
        transform.append('OCR languages in use: '+ocr['lang']+('; requested but not installed: '+'+'.join(ocr['missing']) if ocr['missing'] else ''))
    n_convert=min(doc.page_count,opt.max_pages or doc.page_count)
    independent, independent_lines, independent_error = independent_page_texts(src, n_convert)
    if independent_error: transform.append('independent reference (pdfminer.six) incomplete: '+independent_error)
    vocab=hyphen_vocabulary(doc[i].get_text('text') or '' for i in range(n_convert))
    column_hint=document_columns(doc, n_convert)
    if column_hint: transform.append(f'document-level two-column layout inferred from {column_hint["pages"]} page(s); applied to pages with a short column')
    for i in range(n_convert):
        # v3: an error on one page is contained. The page becomes a recorded gap
        # and the rest of the document is still converted.
        try:
            page=doc[i]; blocks=page.get_text('dict')['blocks']; raw=page.get_text('text') or ''
            table_checks=[]; page_gaps=[]; page_risks=[]; text=raw; ocr_conf=None; parts=[]; layer=None; independent_prose=None; rotated=False; tables_undetected=False
            ocr_result={}
            def page_ocr():
                # OCR is run at most once per page, whoever asks for it.
                if 'r' not in ocr_result: ocr_result['r']=run_ocr_on_page(page, ocr['lang'])
                return ocr_result['r']

            # ---- Page classification ------------------------------------------------
            # blank      no text, no images, no drawings: noted, not OCRed, no gap.
            # scanned    the page has no text layer at all; or OCR EVIDENCE shows that
            #            its text layer holds only a small part of the text visible
            #            on the page image (a stamp or header added on top of a scan).
            # native     everything else. A page is never declared a scan merely
            #            because it has few native tokens on a large image: native
            #            text is kept, and what cannot be decided is recorded.
            coverage=image_coverage(page)
            n_native=len(words(raw))
            is_blank = False
            if not raw.strip() and not page.get_images():
                try: is_blank = not page.get_drawings()
                except Exception: is_blank = False
            is_scanned = not is_blank and not raw.strip()
            overlay=''

            # Hybrid page: a text layer on a page that is mostly image. A second
            # extractor reads the very same layer and cannot vouch for it; our own
            # OCR of the page image can.
            if not is_blank and not is_scanned and coverage >= IMAGE_BACKED:
                undecided=None          # why OCR could not settle what the image holds
                # OCR is consulted when the layer is to be verified, and in any case
                # when the layer is so short (< 20 tokens) that the page may be a scan.
                if ocr['lang'] and (opt.verify_text_layer or n_native < 20):
                    check_text, check_conf, check_error = page_ocr()
                    n_ocr=len(comparable_tokens(check_text))
                    agreement=None if check_error else text_agreement(raw, check_text)
                    if check_error:
                        layer={'state':'unverified','agreement':None}
                        undecided=f'OCR failed: {check_error}'
                    elif check_conf < opt.ocr_min_conf:
                        # OCR is not reliable enough to overrule the native layer.
                        verified=opt.verify_text_layer and agreement >= opt.layer_threshold
                        layer={'state':'verified' if verified else 'unverified','agreement':agreement,'ocr_conf':check_conf}
                        if not verified: undecided=f'OCR confidence {check_conf} is below the minimum of {opt.ocr_min_conf:g}'
                    elif n_ocr >= 20 and n_native < SPARSE_LAYER*n_ocr:
                        # OCR evidence: it reads far more text on the page image than the
                        # layer contains. The page is a scan with an overlay; the overlay
                        # text is kept (see below) if OCR did not read it.
                        is_scanned=True; overlay=raw
                        transform.append(f'page {i+1}: text layer holds {n_native} token(s) but OCR reads {n_ocr} on the page image; page treated as scanned, native text kept')
                    elif not opt.verify_text_layer:
                        layer={'state':'unverified','agreement':None}
                    else:
                        state='verified' if agreement >= opt.layer_threshold else 'disagree'
                        layer={'state':state,'agreement':agreement,'ocr_conf':check_conf}
                        if state=='disagree':
                            transform.append(f'page {i+1}: embedded text layer kept, but it agrees only {agreement:.1%} with OCR of the page image (mean OCR confidence {check_conf})')
                else:
                    layer={'state':'unverified','agreement':None}
                    undecided=ocr['error'] or 'text-layer verification is switched off'
                if undecided and n_native < 20:
                    # A scan carrying a stamp cannot be told from a picture page with a
                    # caption. The native text is kept and the page is left unresolved.
                    page_gaps.append(f'[Context gap: page {i+1} is a page-sized image with only {n_native} token(s) of text layer, which are kept; the image may hold text that was not extracted ({undecided})]')

            if is_scanned:
                if ocr['lang']:
                    text, ocr_conf, ocr_error = page_ocr()
                    if ocr_error:
                        page_gaps.append(f'[Context gap: page {i+1} is scanned and OCR failed ({ocr_error})]'); text=''; ocr_conf=None
                    else:
                        transform.append(f'page {i+1}: OCR executed on scanned page (lang={ocr["lang"]}, mean confidence {ocr_conf})')
                else:
                    page_gaps.append(f'[Context gap: page {i+1} is scanned; text not extractable ({ocr["error"]})]'); text=''

            indicators=re.findall(r'[\ufffd\ue000-\uf8ff]|\(cid:\d+\)',raw)
            if raw and len(indicators)/len(raw)>=0.01: page_risks.append(f'page {i+1}: garbled-encoding indicators ({100*len(indicators)/len(raw):.1f}%)')

            if opt.markers: parts.append(f'<!-- Page {i+1} -->')
            page_tables=[]
            if is_blank:
                parts.append(f'<!-- Blank page {i+1} -->')
                transform.append(f'page {i+1}: blank page (no text, images or drawings)')
            elif ocr_conf is not None:
                # OCR page: no geometry available, paragraphs come from the OCR text.
                parts.append(f'<!-- OCR page {i+1} conf={ocr_conf}{" LOW" if ocr_conf<opt.ocr_min_conf else ""} -->')
                parts.extend(ocr_paragraphs(text, vocab))
                # Overlay text of a scan-with-overlay page is kept if OCR did not read it
                # (invisible text, or text OCR missed), so no native token is dropped.
                if overlay.strip():
                    _, lost, allowed = token_loss(comparable_tokens(overlay), comparable_tokens(text), opt.qc_threshold)
                    if lost > allowed:
                        parts.append(f'<!-- Text layer of page {i+1} (not read by OCR) -->')
                        parts.extend(md_escape(norm(x, vocab)) for x in overlay.splitlines() if x.strip())
                        text=text+'\n'+overlay
            elif not is_scanned:
                # Native-text page: one path for every layout.
                # Table cells and text lines are both in PyMuPDF page coordinates.
                to_fitz=lambda b: tuple(b)
                # A rotated page (/Rotate) is read upright by PyMuPDF, but the
                # independent extractor works in the unrotated frame, so its text
                # order does not apply. Such a page is converted as prose and reported
                # as not independently verified instead of being failed.
                rotated=page.rotation != 0
                if rotated:
                    independent[i]=None
                    page_risks.append(f'page {i+1}: page is rotated ({page.rotation} degrees); tables were not detected and the page was not independently verified')
                # v3: PyMuPDF's table finder, on every page. Unruled tables leave no
                # vector geometry, so there is no cheap way to rule a page out.
                if opt.quality!='fast' and not rotated:
                    page_tables, table_error = find_page_tables(page)
                    if table_error:
                        tables_undetected=True
                        transform.append(f'page {i+1}: table detection unavailable ({table_error}); tables, if any, are written as plain text')
                table_boxes=[to_fitz(t.bbox) for t in page_tables]
                def in_table(bx):
                    cx,cy=(bx[0]+bx[2])/2,(bx[1]+bx[3])/2
                    return any(b[0] <= cx <= b[2] and b[1] <= cy <= b[3] for b in table_boxes)
                lines=[e for e in page_lines(blocks) if not in_table(e['bbox'])]
                two_columns=is_two_column(lines, page.rect.width, column_hint)
                fused=side_by_side_lines(lines, page.rect.width, two_columns)
                lines=merge_baselines(lines, page.rect.width, two_columns)
                elements=list(lines); converted=0
                for t in page_tables:
                    try:
                        cells=table_cells(page, t)
                        rendered=table_md(cells, opt.table_style) if cells else ''
                        if not rendered:
                            page_gaps.append(f'[Context gap: table on page {i+1} yielded no usable cells]')
                            continue
                        elements.append({'kind':'table','bbox':to_fitz(t.bbox),'text':rendered})
                        structure=table_structure_check(page, t, rendered, to_fitz)
                        table_checks.append((cells,rendered,structure)); converted+=1
                        if structure['state']=='checked' and structure['wide_gap'] >= 2:
                            page_risks.append(f'page {i+1}: table {converted} has {structure["wide_gap"]} cell(s) whose words are separated by a wide gap; the detected grid may have merged two columns')
                        if structure['state']=='unavailable':
                            page_risks.append(f'page {i+1}: row/column placement of table {converted} could not be verified (word positions do not line up with the detected grid)')
                    except Exception as exc:
                        page_gaps.append(f'[Context gap: table on page {i+1} could not be converted ({type(exc).__name__})]')
                tables_count+=converted
                # Reference prose for the reading-order check on a table page: the
                # independent extractor's lines that lie outside every table.
                if page_tables and independent_lines[i] is not None:
                    dx,dy=reference_shift(page)
                    kept=[]; removed=[]
                    for x0,top,x1,bottom,line_text in independent_lines[i]:
                        (removed if in_table((x0+dx, top+dy, x1+dx, bottom+dy)) else kept).append(line_text)
                    table_tokens=Counter(tok for cells_,_,_ in table_checks for row in cells_ for cell in row if cell is not None for tok in comparable_tokens(cell))
                    # Sanity check: what was removed must be the table text.
                    if not table_tokens or ratio(table_tokens, Counter(comparable_tokens(''.join(removed)))) >= 0.5:
                        independent_prose=''.join(kept)
                if converted: transform.append(f'page {i+1}: converted {converted} table(s)')
                if opt.detect_borderless:
                    hinted=borderless_table_blocks(page, exclude=table_boxes)
                    if hinted:
                        borderless_count+=hinted
                        page_risks.append(f'page {i+1}: {hinted} table-like block(s) of aligned columns were not detected as tables by the table finder; their cells are written as prose and are not table-verified')
                if two_columns: transform.append(f'page {i+1}: two-column reading order applied')
                if fused >= 2 and not page_tables:
                    page_risks.append(f'page {i+1}: {fused} text lines separated by a wide gap were read as single lines inside one reading column (unresolved columns or tabular text); reading order may be wrong')
                ordered=reading_order(elements, page.rect.width, two_columns)

                # Heading: the largest line of the page, if clearly larger than the body text.
                heading=None
                if opt.quality!='fast' and lines:
                    by_size=Counter()
                    for e in lines: by_size[e['size']]+=len(e['text'])
                    body_size=by_size.most_common(1)[0][0]
                    candidates=[e for e in lines if len(e['text'])<140 and e['max_size']>=14 and e['max_size']>1.15*body_size]
                    if candidates: heading=max(candidates, key=lambda e: (e['max_size'], -e['bbox'][1]))
                prefix=None
                if heading is not None:
                    if title is None: title=heading['text']; prefix='# '
                    elif heading['text']!=title: prefix='## '
                    # a repeat of the document title (running header) stays plain text

                run_lines=[]
                def flush():
                    parts.extend(paragraphs(run_lines, vocab, page.rect.width, two_columns)); run_lines.clear()
                for e in ordered:
                    if e['kind']=='table': flush(); parts.append(e['text'])
                    elif e is heading and prefix: flush(); parts.append(prefix+md_escape(e['text']))
                    else: run_lines.append(e)
                flush()
            if not is_blank and not page_gaps and not words(markdown_plain_text('\n\n'.join(parts))) and not text.strip():
                page_gaps.append(f'[Context gap: page {i+1} has no extractable text]')

            if opt.images:
                for j,img in enumerate(page.get_images(full=True),1):
                    try:
                        info=doc.extract_image(img[0]); ext=info['ext']; fname=f'p{i+1}_image{j}.{ext}'
                        asset_dir.mkdir(parents=True,exist_ok=True)
                        (asset_dir/fname).write_bytes(info['image'])
                        # <...> keeps the link valid when the file name has spaces or brackets.
                        parts.append(f'![Image from page {i+1}](<assets/{stem}/{fname}>)')
                        image_count+=1
                    except Exception:
                        page_gaps.append(f'[Context gap: image on page {i+1} could not be extracted]')
            elif page.get_images(): transform.append(f'page {i+1}: figures excluded by policy; graphical content not verified')

            # OCR rescue. Both candidates (layout projection of the native text and
            # OCR of the rendered page) are scored with the SAME metric against the
            # SAME reference: 4-gram coverage of the page's native text layer.
            # OCR replaces the native projection only if it scores higher and its
            # confidence is acceptable; otherwise the native text is kept.
            source_words = comparable_tokens(text)
            projected_words = comparable_tokens(markdown_plain_text('\n\n'.join(parts)))
            if ocr['lang'] and not is_scanned and not page_tables and len(source_words) >= 20:
                initial_ratio = ratio(grams(source_words), grams(projected_words))
                if initial_ratio < 0.90:
                    ocr_text, ocr_c, ocr_error = page_ocr()
                    ocr_ratio = ratio(grams(source_words), grams(comparable_tokens(ocr_text)))
                    if ocr_error:
                        transform.append(f'page {i+1}: OCR rescue attempted but OCR failed ({ocr_error}); native text kept')
                    elif ocr_ratio > initial_ratio and ocr_c >= opt.ocr_min_conf:
                        kept=[p for p in parts if p.startswith('![Image from page')]
                        parts=([f'<!-- Page {i+1} -->'] if opt.markers else [])
                        parts.append(f'<!-- OCR page {i+1} conf={ocr_c} (rescue) -->')
                        parts.extend(ocr_paragraphs(ocr_text, vocab)); parts.extend(kept)
                        text = ocr_text
                        ocr_conf = ocr_c
                        # The layout warning described the discarded projection.
                        page_risks=[r for r in page_risks if 'inside one reading column' not in r]
                        transform.append(f'page {i+1}: OCR rescue accepted; 4-gram coverage of the native text layer: layout projection {initial_ratio:.1%}, OCR {ocr_ratio:.1%} (mean confidence {ocr_c})')
                    else:
                        transform.append(f'page {i+1}: OCR rescue rejected; 4-gram coverage of the native text layer: layout projection {initial_ratio:.1%}, OCR {ocr_ratio:.1%} (mean confidence {ocr_c}); native text kept')
                    if ocr_conf is None:
                        page_risks.append(f'page {i+1}: layout projection reproduces only {initial_ratio:.1%} of the native text 4-grams; reading order or content may differ from the source')

            parts.extend(page_gaps)
            all_risks.extend(page_risks)
            gap_entries.extend(page_gaps)
            pages.append({'body':'\n\n'.join(parts),'source':norm(text),'gaps':page_gaps,'risk':page_risks,'ocr_conf':ocr_conf,'tables':table_checks,'independent_prose':independent_prose,'scanned':bool(is_scanned),'blank':is_blank,'rotated':rotated,'tables_undetected':tables_undetected,'layer':layer,'independent':independent[i]})
        except Exception as exc:
            gap=f'[Context gap: page {i+1} could not be converted ({type(exc).__name__}: {(str(exc).strip().splitlines() or [""])[0][:120]})]'
            gap_entries.append(gap)
            pages.append({'body':(f'<!-- Page {i+1} -->\n\n' if opt.markers else '')+gap,'source':'','gaps':[gap],'risk':[],'ocr_conf':None,
                          'tables':[],'independent_prose':None,'scanned':False,'blank':False,'rotated':False,'layer':None,
                          'independent':independent[i],'error':True,'tables_undetected':False})
    def page_list(numbers): return ', '.join(map(str,numbers[:12]))+(' ...' if len(numbers) > 12 else '')
    unverified_layer=[k+1 for k,p in enumerate(pages) if p['layer'] and p['layer']['state']=='unverified']
    if unverified_layer:
        all_risks.append(f'{len(unverified_layer)} image-backed page(s) carry an embedded text layer that was not verified against OCR (OCR off, unavailable, failed or low confidence): pages {page_list(unverified_layer)}')
    return pages, title, gap_entries,all_risks,transform,image_count,tables_count,borderless_count

def qc(md_path,pages,images,qc_threshold=0.88,order_threshold=0.70,ocr_min_conf=60.0,layer_threshold=0.70,table_threshold=0.95,block_threshold=0.80,block_min_blocks=2):
    """Run every check and keep the findings apart by what they mean.

    issues               a check ran and found content loss (-> verification FAILED)
    independent[...]:
      'gaps'             content that was not extracted            (extraction INCOMPLETE)
      'low_conf'         OCR pages below the confidence minimum    (extraction LOW_CONFIDENCE)
      'disputes'         a check ran and disagreed: reading order, embedded
                         text layer vs OCR, table cell placement   (verification DISPUTED)
      'not_verified'     {reason: [pages]} - content for which NO independent
                         check was available                       (verification UNVERIFIED)
      'content_pages', 'verified_pages'   non-blank pages / pages independently
                         checked with no dispute and nothing left unverified
    unresolved = gaps + low_conf + disputes (kept for the error listing)."""
    parsed=md_path.read_text(encoding='utf-8'); issues=[]
    if not parsed.strip(): issues.append('[QC gap: output empty]')
    try:
        _, front, body=parsed.split('---',2)
        yaml.safe_load(front)
    except Exception: issues.append('[QC gap: invalid YAML]')
    ratios=[]; gaps=[]; low_conf=[]; disputes=[]; not_verified={}
    independent={'token':[], 'precision':[], 'order':[], 'order_method':[], 'unverified':[], 'ocr':[], 'layer':[], 'layer_unverified':[], 'tables':[], 'tables_unverified':[], 'order_skipped':[], 'blocks':[],
                 'content_pages':0, 'verified_pages':0}
    for i,p in enumerate(pages,1):
        gaps.extend(p['gaps'])
        clean=True                      # stays True only if the page is fully, independently verified
        no_text=False
        def unverified(reason):
            not_verified.setdefault(reason, []).append(i)
        if p.get('error'):
            # The page could not be converted at all: its gap is the finding.
            unverified('page that could not be converted')
            independent['content_pages']+=1; ratios.append((i,0.0))
            continue
        if p.get('tables_undetected'):
            clean=False; unverified('table detection was unavailable: tables, if any, were written as plain text')
        # Embedded text layer over a page image: judged by agreement with OCR.
        if p.get('layer'):
            if p['layer']['agreement'] is not None and p['layer']['state']!='unverified':
                independent['layer'].append((i,p['layer']['agreement']))
            if p['layer']['state']=='unverified':
                independent['layer_unverified'].append(i); clean=False
                unverified('text layer over a page image, not checked against OCR (both extractors read the same layer)')
            elif p['layer']['state']=='disagree':
                clean=False
                disputes.append(f'[QC disagreement: page {i} embedded text layer agrees {p["layer"]["agreement"]:.1%} with OCR of the page image (< {layer_threshold:.1%})]')
        # OCR pages cannot be checked against a second extractor; they are held
        # to a minimum mean OCR confidence, which is a quality floor, not a verification.
        if p['ocr_conf'] is not None:
            independent['ocr'].append((i,p['ocr_conf']))
            if p['ocr_conf'] < ocr_min_conf:
                low_conf.append(f'[Low OCR confidence: page {i} mean confidence {p["ocr_conf"]} (< {ocr_min_conf:g})]')
        # Independent check: converted page vs. pdfminer.six text of the same page.
        chk=independent_check(p, qc_threshold, order_threshold, block_threshold, block_min_blocks)
        if chk['state']=='checked':
            independent['token'].append((i,chk['token']))
            if chk['missing'] > chk['allowed']:
                clean=False
                issues.append(f'[QC gap: page {i} independent token coverage {loss_text(chk["token"],chk["missing"],chk["n"],qc_threshold)}]')
            # Reverse direction: output text the independent reference does not have.
            independent['precision'].append((i,chk['found']))
            if chk['extra'] > chk['extra_allowed']:
                clean=False
                own=Counter(comparable_tokens(p['source']))
                surplus=Counter(comparable_tokens(markdown_plain_text(p['body'])))-Counter(comparable_tokens(p.get('independent') or ''))
                backed=sum((surplus & own).values())
                disputes.append(f'[QC disagreement: page {i} has {chk["extra"]} of {chk["n_out"]} output tokens that the independent extractor does not report ({chk["found"]:.1%} found); '
                                f'{backed} of them are in the text the page was built from, {chk["extra"]-backed} are in neither; e.g. {", ".join(chk["examples"])}]')
            if chk['order'] is not None:
                detail=chk['order_detail']
                independent['order'].append((i,chk['order'])); independent['order_method'].append((i,detail['method']))
                if not detail['ok']:
                    clean=False
                    scores=f'{detail["score"]:.1%} (minimum {detail["minimum"]:.1%})'
                    if detail['sequence'] is not None and detail['method']!='sequence':
                        scores+=f', sequence {detail["sequence"]:.1%} (minimum {detail["sequence_min"]:.1%})'
                    disputes.append(f'[QC disagreement: page {i} reading-order agreement with the independent extractor {scores}; {detail["method"]} comparison]')
            blocks=chk.get('blocks')
            if blocks:
                independent['blocks'].append((i,blocks['score']))
                if not blocks['ok']:
                    clean=False
                    disputes.append(f'[QC disagreement: page {i} block order: {blocks["score"]:.1%} of {blocks["blocks"]} located blocks follow the reference order (minimum {blocks["threshold"]:.0%}), {blocks["overlaps"]} overlapping block pair(s); whole paragraphs or columns may be displaced]')
            if chk['order_skipped']: independent['order_skipped'].append(i)
        elif chk['state']=='no_reference':
            independent['unverified'].append(i); clean=False
            unverified('rotated page: the independent extractor does not apply' if p.get('rotated')
                       else 'OCR page: no independent reference exists (OCR confidence only)' if p['ocr_conf'] is not None
                       else 'the independent extractor returned no text for the page')
        else:
            clean=False; no_text=True   # no text on either side: nothing to verify
        # Tables: token check against the extracted cells, and row/column
        # placement against PyMuPDF word positions.
        for k,(rows,rendered,structure) in enumerate(p.get('tables',[]),1):
            tscore=table_token_coverage(rows,rendered)
            if tscore < qc_threshold:
                clean=False
                issues.append(f'[QC gap: page {i} table-cell token coverage {tscore:.1%} (< {qc_threshold:.1%})]')
            cell_tokens=[t for row in rows for cell in row if cell is not None for t in comparable_tokens(cell)]
            _, t_extra, t_allowed, t_examples = token_additions(comparable_tokens(markdown_plain_text(rendered)), cell_tokens, qc_threshold)
            if t_extra > t_allowed:
                clean=False
                disputes.append(f'[QC disagreement: page {i} table {k} holds {t_extra} token(s) that are not in the extracted cells; e.g. {", ".join(t_examples)}]')
            if structure['state']=='checked':
                independent['tables'].append((i,structure['agreement']))
                if structure['agreement'] < table_threshold:
                    clean=False
                    disputes.append(f'[QC disagreement: page {i} table {k}: {structure["matching"]} of {structure["cells"]} cells hold the words located in that cell ({structure["agreement"]:.1%} < {table_threshold:.1%}); rows or columns may be misplaced]')
            else:
                independent['tables_unverified'].append(i); clean=False
                unverified('table whose row/column placement could not be checked')
        # Internal check: the page text the body was built from (native text
        # layer, or OCR text) against the body, tables included. The criterion
        # is the same for every page length (see allowed_tokens).
        a=comparable_tokens(p['source'])
        b=comparable_tokens(markdown_plain_text(p['body']))
        score, missing, allowed = token_loss(a, b, 1-INTERNAL_TOLERANCE)
        ratios.append((i,score))
        if missing > allowed:
            clean=False
            issues.append(f'[QC gap: page {i} internal token coverage {loss_text(score,missing,len(a),1-INTERNAL_TOLERANCE)}]')
        # Reverse direction, also on OCR pages: output text that is not in the
        # page text the body was built from (duplication, or added content).
        # Body and page text come from the same extractor, so there is no extractor
        # disagreement to allow for: the tolerance here is 1% of the page's tokens.
        _, b_extra, b_allowed, b_examples = token_additions(b, a, 1-INTERNAL_ADDITION_TOLERANCE)
        if b_extra > b_allowed:
            clean=False
            disputes.append(f'[QC disagreement: page {i} has {b_extra} of {len(b)} output tokens that are not in the page text it was built from (repeated or added text); e.g. {", ".join(b_examples)}]')
        # A page with an extraction gap is never counted as verified: content that
        # was not extracted cannot have been checked.
        if p['gaps']: unverified('page with an extraction gap: content that was not extracted could not be verified')
        if not p['blank'] and not (no_text and not p['gaps']):
            independent['content_pages']+=1
            independent['verified_pages']+=clean and not p['gaps']
    source=words(' '.join(p['source'] for p in pages))
    output=words(' '.join(markdown_plain_text(p['body']) for p in pages))
    prov=ratio(grams(output),grams(source)) if len(output)>3 else 1.0
    if any(not p['source'].strip() and not p['gaps'] and not p['blank'] for p in pages): issues.append('[QC gap: unverified blank extracted page]')
    for image in images:
        if not image.exists(): issues.append(f'[QC gap: missing image {image.name}]')
    independent.update(gaps=gaps, low_conf=low_conf, disputes=disputes, not_verified=not_verified)
    return issues,ratios,prov,gaps+low_conf+disputes,independent

def status_model(issues, independent):
    """Two separate claims, and the overall status derived from them.

    extraction    COMPLETE        all content was extracted and serialized
                  LOW_CONFIDENCE  extracted, but OCR itself reports low confidence
                  INCOMPLETE      some content was not extracted (gaps)
    verification  VERIFIED        every non-blank page and every table was checked
                                  against the source by an independent method, and passed
                  UNVERIFIED      nothing failed, but part of the content had no
                                  independent check available
                  DISPUTED        an independent check ran and disagreed
                  FAILED          an independent or internal check found content loss
    status        PASS        = COMPLETE and VERIFIED
                  UNVERIFIED  = COMPLETE and UNVERIFIED
                  PARTIAL     = extraction not COMPLETE, or verification DISPUTED
                  FAILED      = verification FAILED"""
    extraction='INCOMPLETE' if independent['gaps'] else 'LOW_CONFIDENCE' if independent['low_conf'] else 'COMPLETE'
    verification=('FAILED' if issues else 'DISPUTED' if independent['disputes']
                  else 'UNVERIFIED' if independent['not_verified'] else 'VERIFIED')
    status=('FAILED' if verification=='FAILED' else 'PARTIAL' if extraction!='COMPLETE' or verification=='DISPUTED'
            else 'UNVERIFIED' if verification=='UNVERIFIED' else 'PASS')
    return status, extraction, verification

def page_ranges(numbers):
    """[1,2,3,7,9,10] -> '1-3, 7, 9-10'"""
    out=[]; numbers=sorted(set(numbers))
    for n in numbers:
        if out and n==out[-1][1]+1: out[-1][1]=n
        else: out.append([n,n])
    return ', '.join(str(a) if a==b else f'{a}-{b}' for a,b in out)

STATUS_SUFFIX={'PASS':'', 'UNVERIFIED':'.UNVERIFIED', 'PARTIAL':'.PARTIAL', 'FAILED':'.FAILED'}
# v2: the manifest lives in the output folder; its name is reserved so a source
# PDF called conversion_manifest.pdf cannot have its output overwritten by it.
MANIFEST_NAME='conversion_manifest.md'
MANIFEST_STEM=Path(MANIFEST_NAME).stem

def markdown_page(markdown, page_number):
    """Text of one page of a converter output, located by its page marker; None
    if the page has no marker. The Conversion Notes after the last page are
    not part of the page."""
    marks=list(re.finditer(r'(?m)^<!-- Page (\d+) -->[ \t]*$', markdown))
    for k,mark in enumerate(marks):
        if int(mark.group(1))==page_number:
            end=marks[k+1].start() if k+1 < len(marks) else len(markdown)
            notes=markdown.rfind('\n## Conversion Notes', mark.end(), end)
            return markdown[mark.end():(notes if notes != -1 else end)]
    return None

def rendered_tables(markdown_text):
    """Rendered tables of a converted page, in order: pipe tables (a header
    line, a |---| separator, data lines) and <table> blocks."""
    found=[(m.start(), m.group(0)) for m in re.finditer(r'<table>.*?</table>', markdown_text, flags=re.S)]
    found+=[(m.start(), m.group(0).rstrip('\n')) for m in re.finditer(r'(?m)^\|.*\|[ \t]*\n\|(?:\s*:?-{3,}:?\s*\|)+[ \t]*(?:\n\|.*\|[ \t]*)*', markdown_text)]
    return [text for _,text in sorted(found)]

_REFERENCE_CACHE={}
def reference_lines(pdf_path, page_number=None):
    """Independent reference lines, parsed once up to the deepest page asked for.

    audit_page() used to call independent_page_texts(pdf, page_number) on every
    sampled page: auditing pages 20, 70 and 119 of one book parsed 209 pages in
    total, and the last page alone took 9.35 s. pdfminer can only parse forward,
    so the reference is parsed once, up to the deepest page sampled, and reused;
    a deeper request extends the cache.
    """
    key=str(Path(pdf_path).resolve())
    upto=max(1, int(page_number or 1))
    cached=_REFERENCE_CACHE.get(key)
    if cached and cached[0] >= upto: return cached[1], cached[2], cached[3]
    texts, lines, error = independent_page_texts(key, upto)
    _REFERENCE_CACHE[key]=(upto, texts, lines, error)
    return texts, lines, error

def audit_page(pdf_path, page_number, converted, opt=None):
    """Re-verify ONE converted page from the files alone: the source PDF and the
    page text read back from the written Markdown. Applies the same model and
    the same criteria as the conversion QC:

      loss        reference tokens missing from the page       -> FAILED
      additions   page tokens absent from the reference, or
                  from the page's own text layer               -> DISPUTED
      order       length-aware reading-order comparison on the
                  prose (table text removed from both sides)   -> DISPUTED
      tables      every rendered cell must hold the words
                  located in that cell of the detected grid    -> DISPUTED
      text layer  on an image-backed page, agreement with OCR  -> DISPUTED
      no check    OCR page, rotated page, no reference text,
                  unchecked text layer, uncheckable table      -> UNVERIFIED

    Returns {'verification': VERIFIED | UNVERIFIED | DISPUTED | FAILED | BLANK,
             'findings': [text, ...], 'metrics': {name: text}, 'reference': text}."""
    opt=opt or Options()
    findings={'FAILED':[], 'DISPUTED':[], 'UNVERIFIED':[]}; metrics={}
    def result(reference=''):
        level=next((k for k in ('FAILED','DISPUTED','UNVERIFIED') if findings[k]), 'VERIFIED')
        return {'verification': level, 'findings': [f'{k}: {x}' for k in findings for x in findings[k]], 'metrics': metrics, 'reference': reference}
    if re.search(r'<!-- Blank page \d+ -->', converted):
        return {'verification':'BLANK','findings':['blank page: nothing to verify'],'metrics':metrics,'reference':''}
    out_tokens=comparable_tokens(markdown_plain_text(converted))
    ocr_mark=re.search(r'<!-- OCR page \d+ conf=([\d.]+)', converted)
    if ocr_mark:
        metrics['OCR mean confidence']=ocr_mark.group(1)
        if float(ocr_mark.group(1)) < opt.ocr_min_conf: findings['UNVERIFIED'].append(f'OCR confidence {ocr_mark.group(1)} is below the minimum of {opt.ocr_min_conf:g} (extraction LOW_CONFIDENCE)')
        if not re.search(r'\(rescue\) -->', converted):
            findings['UNVERIFIED'].append('OCR page: no independent reference exists (OCR confidence only)')
            return result()
    with pymupdf.open(pdf_path) as doc:
        page=doc[page_number-1]
        if page.rotation != 0:
            findings['UNVERIFIED'].append(f'rotated page ({page.rotation} degrees): the independent extractor does not apply')
            return result()
        _, line_sets, error = reference_lines(pdf_path, page_number)
        ref_lines=line_sets[page_number-1] if not error and len(line_sets) >= page_number else None
        reference=''.join(t[4] for t in ref_lines) if ref_lines else ''
        ref_tokens=comparable_tokens(reference)
        if not ref_tokens:
            findings['UNVERIFIED'].append('the independent extractor returned no text for the page')
            return result(reference)
        # Loss and additions against the independent reference.
        coverage, missing, allowed = token_loss(ref_tokens, out_tokens, opt.qc_threshold)
        metrics['Source tokens found in output']=f'{coverage:.1%} ({missing} of {len(ref_tokens)} missing, {allowed} allowed)'
        if missing > allowed: findings['FAILED'].append(f'{missing} of {len(ref_tokens)} reference tokens are missing ({allowed} allowed)')
        found, extra, extra_allowed, examples = token_additions(out_tokens, ref_tokens, opt.qc_threshold)
        metrics['Output tokens found in source']=f'{found:.1%} ({extra} of {len(out_tokens)} not in the reference, {extra_allowed} allowed)'
        if extra > extra_allowed: findings['DISPUTED'].append(f'{extra} of {len(out_tokens)} output tokens are not in the independent reference; e.g. {", ".join(examples)}')
        # Additions against the page's own text layer (not applicable to an OCR-rescued page).
        if not ocr_mark:
            own=comparable_tokens(page.get_text('text') or '')
            own_cov, own_missing, own_loss_allowed = token_loss(own, out_tokens, 1-INTERNAL_TOLERANCE)
            metrics['Page text-layer tokens found in output']=f'{own_cov:.1%} ({own_missing} of {len(own)} missing, {own_loss_allowed} allowed)'
            if own_missing > own_loss_allowed: findings['FAILED'].append(f'{own_missing} of {len(own)} tokens of the page text layer are missing ({own_loss_allowed} allowed)')
            _, own_extra, own_allowed, own_examples = token_additions(out_tokens, own, 1-INTERNAL_TOLERANCE)
            metrics['Output tokens absent from the page text layer']=f'{own_extra} ({own_allowed} allowed)'
            if own_extra > own_allowed: findings['DISPUTED'].append(f'{own_extra} output tokens are not in the page text layer (repeated or added text); e.g. {", ".join(own_examples)}')
        # Tables: pair each detected table with the rendered table sharing most tokens.
        to_fitz=lambda b: tuple(b)
        tables=rendered_tables(converted); prose=converted
        for rendered in tables: prose=prose.replace(rendered, ' ')
        detected=find_page_tables(page)[0] if tables else []
        boxes=[]; unmatched=list(tables)
        for k,table in enumerate(detected,1):
            cells=Counter(t for row in table_cells(page, table) for cell in row if cell for t in comparable_tokens(cell))
            best=max(unmatched, key=lambda r: ratio(cells, Counter(comparable_tokens(markdown_plain_text(r)))), default=None)
            if best is None or not cells: continue
            unmatched.remove(best); boxes.append(to_fitz(table.bbox))
            structure=table_structure_check(page, table, best, to_fitz)
            if structure['state']=='checked':
                metrics[f'Table {k} cells holding the words located there']=f'{structure["matching"]} of {structure["cells"]} ({structure["agreement"]:.1%})'
                if structure['agreement'] < opt.table_threshold: findings['DISPUTED'].append(f'table {k}: {structure["matching"]} of {structure["cells"]} cells hold the words located in that cell; rows or columns may be misplaced')
            else:
                findings['UNVERIFIED'].append(f'table {k}: row/column placement could not be checked')
        if unmatched: findings['UNVERIFIED'].append(f'{len(unmatched)} rendered table(s) could not be matched to a table detected in the source')
        # Reading order on the prose: table text removed from both sides.
        dx,dy=reference_shift(page)
        inside=lambda l: any(b[0] <= (l[0]+l[2])/2+dx <= b[2] and b[1] <= (l[1]+l[3])/2+dy <= b[3] for b in boxes)
        prose_reference=''.join(l[4] for l in ref_lines if not inside(l))
        order=order_agreement(comparable_tokens(prose_reference), comparable_tokens(markdown_plain_text(prose)), opt.order_threshold, opt.qc_threshold)
        if order:
            text=f'{order["score"]:.1%} (minimum {order["minimum"]:.1%}, {order["method"]})'
            if order['sequence'] is not None and order['method']!='sequence': text+=f'; sequence {order["sequence"]:.1%} (minimum {order["sequence_min"]:.1%})'
            metrics['Reading-order agreement']=text
            if not order['ok']: findings['DISPUTED'].append(f'reading order disagrees with the independent extractor: {text}')
        blocks=block_order(comparable_tokens(prose_reference), prose, opt.block_order_threshold, opt.block_order_min_blocks)
        if blocks:
            metrics['Block order']=f'{blocks["score"]:.1%} of {blocks["blocks"]} blocks in reference order (minimum {blocks["threshold"]:.0%}), {blocks["overlaps"]} overlapping pair(s)'
            if not blocks['ok']:
                findings['DISPUTED'].append(f'block order disagrees with the independent extractor: {blocks["score"]:.1%} of {blocks["blocks"]} blocks in order, {blocks["overlaps"]} overlapping block pair(s)')
        # Text layer over a page image: only OCR can vouch for it.
        if not ocr_mark and image_coverage(page) >= IMAGE_BACKED:
            ocr=ocr_setup(opt)
            checked=run_ocr_on_page(page, ocr['lang']) if ocr['lang'] and opt.verify_text_layer else None
            if checked is None or checked[2] or checked[1] < opt.ocr_min_conf:
                findings['UNVERIFIED'].append('text layer over a page image, not checked against OCR (OCR off, unavailable, failed or low confidence)')
            else:
                agreement=text_agreement(page.get_text('text') or '', checked[0])
                metrics['Text layer agreement with OCR']=f'{agreement:.1%} (minimum {opt.layer_threshold:.1%})'
                if agreement < opt.layer_threshold: findings['DISPUTED'].append(f'the embedded text layer agrees only {agreement:.1%} with OCR of the page image')
        return result(reference)

def output_source(path):
    """Source PDF recorded in the front matter of an existing output file, or
    None when the file is not a converter output (no readable front matter)."""
    try:
        with Path(path).open(encoding='utf-8') as stream: head=stream.read(8192)
        if not head.startswith('---'): return None
        data=yaml.safe_load(head.split('---',2)[1])
        return str(data['source']) if isinstance(data, dict) and data.get('source') else None
    except Exception:
        return None

def run(paths,output_dir,opt):
    opt.resolve(); out=Path(output_dir).expanduser().resolve(); out.mkdir(parents=True,exist_ok=True)
    probe=out/'.pdf2md_write_test'; probe.write_text('ok'); probe.unlink()
    reports=[]; used=set()
    for original in paths:
        src=Path(original).expanduser().resolve()
        rec={'input':str(src),'status':'SKIPPED','errors':[]}
        if src in used: rec['errors'].append('duplicate input path'); reports.append(rec); continue
        used.add(src)
        try:
            if not src.is_file() or src.stat().st_size==0: raise ValueError('unreadable or empty input')
            with src.open('rb') as stream:
                if stream.read(5) != b'%PDF-':
                    raise ValueError('invalid PDF header')
            with pymupdf.open(src) as doc:
                if doc.needs_pass: raise ValueError('encrypted / password protected')
                # One output per input, named by status:
                #   <stem>.md (PASS), <stem>.UNVERIFIED.md, <stem>.PARTIAL.md, <stem>.FAILED.md
                # An existing file is replaced only with overwrite=True AND only if its
                # front matter names this same source PDF. A file that belongs to another
                # source (two PDFs with the same name in different folders), or that is
                # not a converter output at all, is never touched: the next free
                # <stem>_<n> is used instead.
                variants=lambda name: [out/f'{name}{suffix}.md' for suffix in STATUS_SUFFIX.values()]
                def previous_outputs(name):
                    legacy=re.compile(re.escape(name)+r'\.(?:UNVERIFIED|PARTIAL|FAIL|FAILED)\.\d+\.md')
                    return [v for v in variants(name) if v.exists()]+[f for f in out.iterdir() if legacy.fullmatch(f.name)]
                def available(name):
                    if name == MANIFEST_STEM: return False
                    if any(v in used for v in variants(name)): return False
                    existing=[v for v in variants(name) if v.exists()]
                    return not existing or (opt.overwrite and all(output_source(v)==str(src) for v in existing))
                stem=src.stem; n=0; skipped=[]
                while not available(stem):
                    skipped+=[f'{v.name} (source: {output_source(v) or "not a converter output"})' for v in variants(stem) if v.exists()]
                    n+=1; stem=f'{src.stem}_{n}'
                used.update(variants(stem))
                # This source's own earlier outputs under that name, and the asset
                # folders they link to: the only things a re-run may delete.
                owned=[f for f in previous_outputs(stem) if output_source(f)==str(src)] if opt.overwrite else []
                owned_assets=set()
                for f in owned:
                    try: owned_assets.update(re.findall(r'\]\(<?assets/(.+?-[0-9a-f]{32})/', f.read_text(encoding='utf-8')))
                    except Exception: pass
                with tempfile.TemporaryDirectory(prefix='.pdf2md-',dir=out) as staging:
                    stage=Path(staging)
                    staged_asset=stage/'assets'/stem
                    pages,title,gaps,risks,changes,img_count,table_count,borderless=extract_pages(doc,src,opt,staged_asset,stem)
                    if skipped: changes.append(f'output named {stem}: the requested name {src.stem}.md was already in use, so the next free name was taken; left untouched: '+'; '.join(skipped))
                    # Classification describes the SOURCE pages (text layer present or not), not the OCR result.
                    n_pages=len(pages); scanned_pages=sum(p['scanned'] for p in pages)
                    classification='TEXT_NATIVE' if scanned_pages==0 else 'SCANNED' if scanned_pages==n_pages else 'MIXED'
                    opts={'source':str(src),'source_sha256':hashlib.sha256(src.read_bytes()).hexdigest(),'pages':f'{n_pages} / {doc.page_count}','class':classification,'quality_tier':opt.quality,'ocr_used':any(p['ocr_conf'] is not None for p in pages),'images_extracted':img_count,'tables_converted':table_count,'borderless_tables_suspected':borderless,'converted':date.today().isoformat(),'gaps':len(gaps),'fidelity_risks':len(risks)}
                    figure_policy=('- Figure policy: embedded raster images are extracted to assets/ and linked; vector graphics are not extracted. Image content is not verified.' if opt.images
                                   else '- Figure policy: graphical figures excluded; their semantic content is not verified. Textual captions are retained when extracted.')
                    notes=['## Conversion Notes',figure_policy,'### Gaps'] + (['- '+x for x in gaps] or ['- None']) + ['### Fidelity risks'] + (['- '+x for x in risks] or ['- None']) + ['### Transformations'] + (['- '+x for x in changes] or ['- None'])
                    notes += ['- Extraction library: PyMuPDF '+importlib.metadata.version('PyMuPDF'), '- Table detector: PyMuPDF find_tables']
                    if opts['ocr_used'] and shutil.which('tesseract'): notes+=['- OCR: Tesseract '+(subprocess.run(['tesseract','--version'],capture_output=True,text=True).stdout.splitlines()[0])]
                    header='---\n'+yaml.safe_dump(opts,sort_keys=False,allow_unicode=True).strip()+'\n---\n'
                    # The title is emitted once: in place on the page where it was
                    # found, or here (file name) when no title was detected.
                    top='' if title else '\n# '+src.stem+'\n'
                    content=header+top+'\n'+ '\n\n'.join(p['body'] for p in pages)+'\n\n'+'\n'.join(notes)+'\n'
                    stage_md=stage/'converted.md'
                    stage_md.write_text(content,encoding='utf-8')
                    assets=list(staged_asset.glob('*')) if staged_asset.exists() else []
                    issues,cover,prov,unresolved,indep=qc(stage_md,pages,assets,opt.qc_threshold,opt.order_threshold,opt.ocr_min_conf,opt.layer_threshold,opt.table_threshold,opt.block_order_threshold,opt.block_order_min_blocks)
                    status, extraction, verification = status_model(issues, indep)
                    # The file states both claims itself: front matter and a Verification note.
                    opts.update(status=status, extraction=extraction, verification=verification,
                                pages_verified=f'{indep["verified_pages"]} / {indep["content_pages"]}')
                    verification_notes=['### Verification',f'- Status: {status} (extraction {extraction}, verification {verification})',
                                        f'- Pages independently verified: {indep["verified_pages"]} of {indep["content_pages"]} non-blank page(s)']
                    verification_notes+=[f'- Not independently verified - {reason}: page(s) {page_ranges(pgs)}' for reason,pgs in indep['not_verified'].items()]
                    notes=notes[:2]+verification_notes+notes[2:]
                    header='---\n'+yaml.safe_dump(opts,sort_keys=False,allow_unicode=True).strip()+'\n---\n'
                    content=header+top+'\n'+ '\n\n'.join(p['body'] for p in pages)+'\n\n'+'\n'.join(notes)+'\n'
                    stage_md.write_text(content,encoding='utf-8')
                    destination=out/(stem+STATUS_SUFFIX[status]+'.md')
                    committed_asset=None
                    if assets:
                        namespace=f'{stem}-{uuid.uuid4().hex}'
                        old_prefix=f'assets/{stem}/'
                        new_prefix=f'assets/{namespace}/'
                        content=content.replace(old_prefix,new_prefix)
                        stage_md.write_text(content,encoding='utf-8')
                        issues_after,_,_,_,_=qc(stage_md,pages,assets,opt.qc_threshold,opt.order_threshold,opt.ocr_min_conf,opt.layer_threshold,opt.table_threshold,opt.block_order_threshold,opt.block_order_min_blocks)
                        if issues_after != issues:
                            raise RuntimeError('QC changed after asset path rewriting')
                        committed_asset=out/'assets'/namespace
                        committed_asset.parent.mkdir(parents=True,exist_ok=True)
                        staged_asset.rename(committed_asset)
                    try:
                        os.replace(stage_md,destination)
                    except Exception:
                        if committed_asset is not None:
                            shutil.rmtree(committed_asset)
                        raise
                    target=destination
                    # Remove this source's own outputs from earlier runs (any status) and
                    # the asset folders they linked to. Ownership was established from
                    # each file's front matter before the conversion started.
                    for old in owned:
                        if old != destination and old.is_file(): old.unlink()
                    for name in owned_assets:
                        old=out/'assets'/name
                        if old != committed_asset and old.is_dir() and re.fullmatch(re.escape(stem)+r'-[0-9a-f]{32}', name): shutil.rmtree(old)
            rec.update(status=status,extraction=extraction,verification=verification,output=str(target),output_id=stem,renamed=skipped,pages=n_pages,classification=classification,ocr=opts['ocr_used'],gaps=len(gaps),risks=len(risks),errors=issues+unresolved,coverage=cover,provenance=prov,independent=indep,bytes=target.stat().st_size,sha256=hashlib.sha256(target.read_bytes()).hexdigest()[:12])
        except Exception as exc:
            rec.update(status='ERROR',errors=[str(exc)])
        reports.append(rec)
    lines=['# PDF → Markdown Conversion Manifest','',f'- Quality tier: {opt.quality} (OCR: {opt.ocr}, images: {opt.images}, page markers: {opt.markers}, table style: {opt.table_style})',f'- Output directory: {out}',f'- Inputs: {len(reports)} | '+' | '.join(f'{k}: {v}' for k,v in sorted(Counter(r["status"] for r in reports).items())),'- Verification scope (internal): text-token and table-cell consistency against the PyMuPDF extraction the output was built from. Loss and additions against the text of the page itself are held to 1%.',f'- Verification scope (independent): each converted page is compared with pdfminer.six text of the same page. Token coverage below {opt.qc_threshold:.0%} fails the document; the criterion is the same for pages of any length (at most floor((1 - threshold) x tokens) may be missing, which is zero on very short pages). The check runs in both directions: output tokens that the reference does not contain, beyond the same tolerance, mark the document PARTIAL with verification DISPUTED - a disagreement between extractors, not proof of added content. Output tokens absent from the page text the output was itself built from are held to a 1% tolerance; reading-order agreement below {opt.order_threshold:.0%} marks it PARTIAL. The order comparison adapts to page length: 4-grams from 20 tokens, 3-grams from 8, 2-grams from 4, and direct sequence comparison (longest common subsequence) on every page under 20 tokens; there the minimums are lowered by what the token criterion already tolerates. On a page with tables the order check covers the prose around them. Pages without a text layer (OCR pages) have no independent reference: they make the document UNVERIFIED at best, and must reach a mean OCR confidence of {opt.ocr_min_conf:g}, otherwise it is PARTIAL.', f'- Verification scope (tables): each rendered cell must hold exactly the words PyMuPDF locates inside that cell of the detected grid; below {opt.table_threshold:.0%} of matching cells the document is PARTIAL. The grid itself (which lines are rows and columns) comes from the PyMuPDF table finder and is not independently verified; cells with column-sized gaps are reported as a risk.', f'- Verification scope (image-backed pages): where a text layer sits on a page scan, both extractors read that same layer, so it is compared with OCR of the page image instead. Agreement below {opt.layer_threshold:.0%} marks the document PARTIAL; if OCR is off, unavailable or below the confidence minimum, the pages are not verified and the document is UNVERIFIED at best.','- Not verified: rendered-page fidelity, excluded figures, and the structure of any table the table finder does not detect. Such a table is read as prose; pages where aligned columns suggest one are counted in borderless_tables_suspected and listed as a fidelity risk.', '- Reading order is checked twice: n-gram agreement (how much text survived, in what local order) and block order (each output paragraph is located in the reference by a unique 6-gram; displaced blocks or interleaved spans mark the document PARTIAL). n-gram agreement alone cannot see whole blocks move.', '- Block order is skipped on pages where fewer than block_order_min_blocks paragraphs have an unambiguous anchor in the reference; the manifest then shows no block-order figure instead of guessing.','',
           '## Status model','- **extraction**: COMPLETE (all content extracted and serialized), LOW_CONFIDENCE (extracted, but OCR reports low confidence), INCOMPLETE (some content not extracted).',
           '- **verification**: VERIFIED (every non-blank page and table checked against the source by an independent method, and passed), UNVERIFIED (nothing failed, but part of the content had no independent check), DISPUTED (a check ran and disagreed), FAILED (a check found content loss).',
           '- **status**: PASS = COMPLETE + VERIFIED; UNVERIFIED = COMPLETE + UNVERIFIED; PARTIAL = extraction not COMPLETE, or verification DISPUTED; FAILED = verification FAILED. The status is also the file suffix.',
           '', '| # | Input | Pages | Output | Status | Extraction | Verification | Verified pages | Gaps | Risks |','|---|---|---:|---|---|---|---|---|---:|---:|']
    for j,r in enumerate(reports,1):
        ind=r.get('independent') or {}
        verified=f'{ind["verified_pages"]} / {ind["content_pages"]}' if ind else '—'
        lines.append(f'| {j} | {Path(r["input"]).name} | {r.get("pages","—")} | {Path(r.get("output", "—")).name} | {r["status"]} | {r.get("extraction","—")} | {r.get("verification","—")} | {verified} | {r.get("gaps","—")} | {r.get("risks","—")} |')
    lines+=['','## Evidence','| Input | Bytes | SHA-256 (first 12) | Lowest internal token coverage (page) | Body text provenance diagnostic |','|---|---:|---|---|---|']
    for r in reports:
        if 'sha256' in r:
            low=min(r['coverage'],key=lambda t:t[1]) if r['coverage'] else None
            lines.append(f'| {Path(r["input"]).name} | {r["bytes"]} | {r["sha256"]} | {f"{low[1]:.1%} (p. {low[0]})" if low else "n/a"} | {r["provenance"]:.1%} |')
    lines+=['','## Independent verification (pdfminer.six reference)','| Input | Pages checked | Lowest share of source found in output (page) | Lowest share of output found in source (page) | Lowest order agreement (page) | Lowest block order (page) | Pages not independently verified | Lowest OCR confidence (page) |','|---|---:|---|---|---|---|---|---|']
    def lowest(pairs):
        if not pairs: return 'n/a'
        page,score=min(pairs,key=lambda t:t[1]); return f'{score:.1%} (p. {page})'
    for r in reports:
        ind=r.get('independent')
        if ind is None: continue
        unverified=', '.join(map(str,ind['unverified'])) or 'none'
        ocr_low=min(ind['ocr'],key=lambda t:t[1]) if ind['ocr'] else None
        lines.append(f'| {Path(r["input"]).name} | {len(ind["token"])} | {lowest(ind["token"])} | {lowest(ind["precision"])} | {lowest(ind["order"])} | {lowest(ind.get("blocks") or [])} | {unverified} | {f"{ocr_low[1]:g} (p. {ocr_low[0]})" if ocr_low else "n/a"} |')
    layered=[r for r in reports if r.get('independent') and (r['independent']['layer'] or r['independent']['layer_unverified'])]
    if layered:
        lines+=['','## Image-backed pages (embedded text layer checked against OCR)','| Input | Pages checked | Lowest agreement with OCR (page) | Pages not verified |','|---|---:|---|---:|']
        for r in layered:
            ind=r['independent']
            lines.append(f'| {Path(r["input"]).name} | {len(ind["layer"])} | {lowest(ind["layer"])} | {len(ind["layer_unverified"])} |')
    tabled=[r for r in reports if r.get('independent') and (r['independent']['tables'] or r['independent']['tables_unverified'])]
    if tabled:
        lines+=['','## Tables (cell placement checked against word positions)','| Input | Tables checked | Lowest cell agreement (page) | Tables not verified | Table pages without order check |','|---|---:|---|---:|---:|']
        for r in tabled:
            ind=r['independent']
            lines.append(f'| {Path(r["input"]).name} | {len(ind["tables"])} | {lowest(ind["tables"])} | {len(ind["tables_unverified"])} | {len(ind["order_skipped"])} |')
    renamed=[r for r in reports if r.get('renamed')]
    if renamed:
        lines+=['','## Output names changed to protect existing files']
        for r in renamed:
            lines.append(f'- {r["input"]} -> {Path(r["output"]).name}; left untouched: '+'; '.join(r['renamed']))
    unchecked=[r for r in reports if r.get('independent') and r['independent']['not_verified']]
    if unchecked:
        lines+=['','## Content not independently verified']
        for r in unchecked:
            for reason,pgs in r['independent']['not_verified'].items():
                lines.append(f'- {Path(r["input"]).name}: {reason} - page(s) {page_ranges(pgs)}')
    lines+=['','## Gaps and Failures']
    for r in reports:
        for e in r['errors']:
            lines.append(f'- {Path(r["input"]).name}: {e}')
    if not any(r['errors'] for r in reports):
        lines.append('- None')
    manifest=out/MANIFEST_NAME
    manifest.write_text('\n'.join(lines)+'\n',encoding='utf-8')
    return reports,manifest

def main():
    ap=argparse.ArgumentParser(description=__doc__); ap.add_argument('pdfs',nargs='+'); ap.add_argument('--output-dir',default='markdown_output'); ap.add_argument('--quality',default='balanced'); ap.add_argument('--ocr',choices=['on','off']); ap.add_argument('--images',choices=['on','off']); ap.add_argument('--page-markers',choices=['on','off']); ap.add_argument('--table-style',choices=['pipe','html'],default='pipe'); ap.add_argument('--max-pages',type=int); ap.add_argument('--overwrite',action='store_true'); ap.add_argument('--qc-threshold',type=float,default=0.88); ap.add_argument('--ocr-lang',default='rus+eng'); ap.add_argument('--order-threshold',type=float,default=0.70); ap.add_argument('--ocr-min-conf',type=float,default=60.0)
    ap.add_argument('--layer-threshold',type=float,default=0.70); ap.add_argument('--table-threshold',type=float,default=0.95); ap.add_argument('--no-verify-text-layer',action='store_true')
    a=ap.parse_args()
    on=lambda v: None if v is None else v=='on'
    opt=Options(quality=a.quality, ocr=on(a.ocr), images=on(a.images), markers=on(a.page_markers), table_style=a.table_style,
                max_pages=a.max_pages, overwrite=a.overwrite, qc_threshold=a.qc_threshold, ocr_lang=a.ocr_lang,
                order_threshold=a.order_threshold, ocr_min_conf=a.ocr_min_conf,
                verify_text_layer=not a.no_verify_text_layer, layer_threshold=a.layer_threshold, table_threshold=a.table_threshold)
    try: reports,manifest=run(a.pdfs,a.output_dir,opt)
    except (ValueError,RuntimeError,OSError) as e: print(f'[Context gap: {e}]',file=sys.stderr); return 2
    print(manifest.read_text()); return 1 if any(r['status']!='PASS' for r in reports) else 0
# Command-line entry point. Inside a notebook kernel this cell only defines the
# converter; call run() from the conversion cell instead.
if __name__=='__main__' and 'ipykernel' not in sys.modules:
    sys.exit(main())

In [ ]:
# 3. Configuration: detect the notebook workspace before searching for PDF files.
import os
import sys
from pathlib import Path

def detect_pdf_input_dir():
    """Return the input workspace for the current notebook environment."""
    if "google.colab" in sys.modules or "COLAB_RELEASE_TAG" in os.environ:
        return Path("/content")
    if "DEEPNOTE_PROJECT_ID" in os.environ:
        return Path("/work")
    # JetBrains Datalore and local Jupyter use the current notebook directory.
    return Path.cwd()

PDF_INPUT_DIR = detect_pdf_input_dir()
PDF_OUTPUT_DIR = PDF_INPUT_DIR / "markdown_output"
PDF_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# Tesseract language packs for OCR, '+'-separated (e.g. "spa+eng").
# Each pack must be installed in cell 1; packs that are missing are
# reported in the Conversion Notes instead of failing silently.
OCR_LANGUAGES = "rus+eng"

print(f"Input folder: {PDF_INPUT_DIR}")
print(f"Output directory: {PDF_OUTPUT_DIR.resolve()}")
print(f"PDF files found: {sum(p.is_file() and p.suffix.lower() == '.pdf' for p in PDF_INPUT_DIR.iterdir())}")

for file in sorted(PDF_OUTPUT_DIR.glob("*.md")):
    print(file.name)

In [ ]:
# 4. Conversion (single run; OCR fallback enabled for scanned pages).
pdf_files = sorted(
    [
        p for p in Path(PDF_INPUT_DIR).iterdir()
        if p.is_file() and p.suffix.lower() == ".pdf"
    ],
    key=lambda p: (p.name.casefold(), p.name)
)

print(f"Input folder: {PDF_INPUT_DIR}")
print(f"PDF files found: {len(pdf_files)}")

if pdf_files:
    options = Options(
        quality="high",
        ocr=True,
        ocr_lang=OCR_LANGUAGES,
        images=False,
        overwrite=True,
        # v3: do not re-OCR pages that already carry a text layer over a scan.
        # Those books are converted from their own layer and marked UNVERIFIED.
        # Set to True to compare the layer with OCR again (about 5 to 9 s per page).
        verify_text_layer=False,
    )

    reports, manifest = run(
        pdf_files,
        PDF_OUTPUT_DIR,
        options
    )

    # Report what actually happened, not an unconditional success message.
    status_counts = Counter(r["status"] for r in reports)
    print(
        "\nConversion finished: "
        + ", ".join(f"{status_counts[s]} {s}" for s in sorted(status_counts))
        + "\n"
    )
    print(manifest.read_text())
else:
    reports, manifest = [], None
    print("No PDF files found.")

In [ ]:
# 5. Sampled audit of the conversion produced by the previous cell.
#
# Every sampled page is re-verified from the files alone (the source PDF and
# the page text read back from the written Markdown) with audit_page(), which
# applies the same verification model and criteria as the conversion QC:
# token loss, content additions, reading order, table cell placement and, on
# image-backed pages, agreement of the text layer with OCR.
from pathlib import Path
from collections import Counter
import random
import pymupdf

# ============================================================
# 1. CONFIGURATION
# ============================================================

NUM_DOCUMENTS = 3
PAGES_PER_DOCUMENT = 3
RANDOM_SEED = 42

rng = random.Random(RANDOM_SEED)

if "reports" not in globals():
    raise RuntimeError(
        "Conversion reports are unavailable. "
        "Execute the PDF conversion cell first."
    )

# The audit uses the options of the conversion it audits, so its criteria
# cannot drift from the ones the outputs were produced under.
audit_options = options if "options" in globals() else Options()
audit_options.resolve()

# v2: the audit locates pages by their '<!-- Page n -->' marker, which the fast
# tier does not write. Say so instead of reporting every page as UNVERIFIED.
if not audit_options.markers:
    raise RuntimeError(
        "The conversion ran with page markers off (quality='fast'), so pages "
        "cannot be located in the output. Re-run the conversion with "
        "quality='balanced' or 'high', or markers=True."
    )

OUTPUT = (
    Path(PDF_OUTPUT_DIR)
    if "PDF_OUTPUT_DIR" in globals()
    else Path(reports[0]["output"]).parent if reports and reports[0].get("output")
    else Path("markdown_output")
)
AUDIT_DIR = OUTPUT / "fidelity_audit"
AUDIT_DIR.mkdir(parents=True, exist_ok=True)

# ============================================================
# 2. CONVERTED OUTPUTS (every status), BY OUTPUT IDENTITY
# ============================================================

# One entry per converted output. The key is the converter's output identity
# (the output name without its status suffix, e.g. "same_1"), which is unique
# in the output folder even when two source PDFs share a file name.
converted_outputs = {
    r["output_id"]: r
    for r in reports
    if r.get("output") and r.get("output_id") and Path(r["output"]).is_file()
    and Path(r["input"]).is_file()
}

if not converted_outputs:
    raise RuntimeError("No converted PDFs were found.")

# ============================================================
# 3. RANDOM DOCUMENT AND PAGE SELECTION
# ============================================================

selected = rng.sample(
    sorted(converted_outputs, key=lambda k: (k.casefold(), k)),
    min(NUM_DOCUMENTS, len(converted_outputs))
)

SAMPLES = {}

for output_id in selected:

    # Only pages that were actually converted can be sampled: with
    # max_pages set, that is fewer than the source document has.
    converted_pages = converted_outputs[output_id].get("pages") or 0

    if converted_pages == 0:
        print(f"SKIP: {output_id} has no converted pages.")
        continue

    SAMPLES[output_id] = sorted(
        rng.sample(
            range(1, converted_pages + 1),
            min(PAGES_PER_DOCUMENT, converted_pages)
        )
    )

print("=" * 72)
print("RANDOM FIDELITY AUDIT SAMPLE")
print("=" * 72)

for output_id, pages in SAMPLES.items():
    r = converted_outputs[output_id]
    print(f"{Path(r['output']).name}  <-  {r['input']}")
    print(
        f"  conversion status: {r['status']} "
        f"(extraction {r['extraction']}, verification {r['verification']}); "
        f"pages converted: {r['pages']}; sampled: {pages}"
    )

print(f"\nRandom seed: {RANDOM_SEED}")
print(f"Documents selected: {len(SAMPLES)}")

# ============================================================
# 4. RE-VERIFY EACH SAMPLED PAGE FROM THE FILES
# ============================================================

results = []

for output_id, page_numbers in SAMPLES.items():

    r = converted_outputs[output_id]
    pdf_path = Path(r["input"])
    md_path = Path(r["output"])
    markdown = md_path.read_text(encoding="utf-8")

    # v2: parse the independent reference once per document, up to the deepest
    # sampled page, instead of once per sampled page.
    reference_lines(pdf_path, max(page_numbers))

    for page_number in page_numbers:

        converted = markdown_page(markdown, page_number)

        if converted is None:
            outcome = {
                "verification": "UNVERIFIED",
                "findings": ["UNVERIFIED: page marker not found in the Markdown (page markers are off)"],
                "metrics": {},
                "reference": ""
            }
            converted = ""
        else:
            outcome = audit_page(
                pdf_path,
                page_number,
                converted,
                audit_options
            )

        results.append({
            "output": md_path.name,
            "page": page_number,
            "verification": outcome["verification"]
        })

        # --------------------------------------------------------
        # Individual report, named by output identity
        # --------------------------------------------------------

        report = (
            "# Fidelity audit\n\n"
            f"Output: {md_path.name}\n\n"
            f"Source: {pdf_path}\n\n"
            f"Page: {page_number} of {r['pages']} converted\n\n"
            f"Conversion status: {r['status']} "
            f"(extraction {r['extraction']}, verification {r['verification']})\n\n"
            f"Random seed: {RANDOM_SEED}\n\n"
            f"Page verification: {outcome['verification']}\n\n"
            "## Findings\n\n"
            + ("\n".join(f"- {x}" for x in outcome["findings"]) or "- None")
            + "\n\n## Measurements\n\n"
            + ("\n".join(f"- {k}: {v}" for k, v in outcome["metrics"].items()) or "- None")
            + "\n\n## Independent reference text (pdfminer.six)\n\n"
            f"```\n{outcome['reference']}\n```\n\n"
            "## Converted Markdown\n\n"
            f"```\n{converted}\n```\n"
        )

        destination = AUDIT_DIR / f"{output_id}_page_{page_number}.md"
        destination.write_text(report, encoding="utf-8")

        print(
            f"{md_path.name} | page {page_number} | "
            f"{outcome['verification']}"
            + "".join(f"\n      {x}" for x in outcome["findings"])
        )

# ============================================================
# 5. SUMMARY
# ============================================================

print("\n" + "=" * 72)
print("FIDELITY AUDIT SUMMARY")
print("=" * 72)

counts = Counter(x["verification"] for x in results)

print(f"Documents sampled: {len(SAMPLES)}")
print(f"Pages audited: {len(results)}")

for level in ("VERIFIED", "UNVERIFIED", "DISPUTED", "FAILED", "BLANK"):
    print(f"{level}: {counts[level]}")

print(f"\nPage reports: {AUDIT_DIR.resolve()}")